In [18]:
%reload_ext google.colab.data_table

In [19]:
# ============================================================
# RLT REIMPLEMENTATION — STEP 1
# Environment & Reproducibility
# ============================================================

import torch
import torch.nn as nn
import torch.nn.functional as F

import math
import random
import numpy as np

# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ------------------------------------------------------------
# Device
# ------------------------------------------------------------

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("PyTorch:", torch.__version__)
print("Device:", device)

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)

    print("GPU:", gpu.name)
    print(
        "VRAM:",
        round(gpu.total_memory / 1024**3, 2),
        "GB"
    )

print("Seed:", SEED)
print("Environment ready.")

PyTorch: 2.11.0+cu128
Device: cuda
GPU: Tesla T4
VRAM: 14.56 GB
Seed: 42
Environment ready.


In [2]:
# ============================================================
# RLT REIMPLEMENTATION — STEP 2
# Experiment Configuration
# ============================================================

# ------------------------------------------------------------
# Reported experimental setup
# ------------------------------------------------------------

TRAIN_LENGTH = 32

EVAL_LENGTHS = [16, 32, 64, 128]

TRAIN_UPDATES = 2000

NUM_SEEDS = 3

SEEDS = [42, 123, 2026]

# Target parameter scale reported in the post
TARGET_PARAMS = 79_000

# Models compared in the reported experiment
MODEL_NAMES = [
    "RLT",
    "GRU",
    "Causal Transformer",
    "Token-only Merge"
]

# Synthetic tasks
TASK_NAMES = [
    "Parity",
    "Five-state transitions"
]

print("RLT REIMPLEMENTATION CONFIGURATION")
print("-----------------------------------")
print("Training length :", TRAIN_LENGTH)
print("Evaluation      :", EVAL_LENGTHS)
print("Training updates:", TRAIN_UPDATES)
print("Number of seeds :", NUM_SEEDS)
print("Seeds           :", SEEDS)
print("Target params   :", f"~{TARGET_PARAMS:,}")
print()
print("Models:")
for name in MODEL_NAMES:
    print(" -", name)

print()
print("Tasks:")
for name in TASK_NAMES:
    print(" -", name)

print()
print("Configuration locked.")

RLT REIMPLEMENTATION CONFIGURATION
-----------------------------------
Training length : 32
Evaluation      : [16, 32, 64, 128]
Training updates: 2000
Number of seeds : 3
Seeds           : [42, 123, 2026]
Target params   : ~79,000

Models:
 - RLT
 - GRU
 - Causal Transformer
 - Token-only Merge

Tasks:
 - Parity
 - Five-state transitions

Configuration locked.


In [3]:
# ============================================================
# RLT REIMPLEMENTATION — STEP 3
# Synthetic State-Tracking Tasks
# ============================================================

def generate_parity(n_samples, length, seed=42):
    """
    Parity state-tracking task.

    Operation tokens:
        0 = no flip
        1 = flip

    Initial state:
        0

    Transition:
        state <- state XOR operation

    Target:
        final state (0 or 1)

    IMPORTANT:
    The state is NOT provided as an input.
    Only operation tokens are given to the model.
    """

    rng = np.random.default_rng(seed)

    operations = rng.integers(
        0,
        2,
        size=(n_samples, length),
        dtype=np.int64
    )

    # Initial state = 0
    final_state = np.bitwise_xor.reduce(
        operations,
        axis=1
    )

    return operations, final_state


def generate_five_state(n_samples, length, seed=42):
    """
    Five-state order-dependent transition task.

    This is an explicit reimplementation choice because
    the public post describes the task but does not give
    the exact transition table.

    States:
        0, 1, 2, 3, 4

    Operation tokens:
        0, 1, 2, 3, 4

    Transition table:
        operation 0: state + 0
        operation 1: state + 1
        operation 2: state + 2
        operation 3: state - 1
        operation 4: state - 2

    Transitions wrap around modulo 5.

    Initial state:
        0

    Target:
        final state (0...4)

    The state itself is NEVER provided as an input.
    """

    rng = np.random.default_rng(seed)

    operations = rng.integers(
        0,
        5,
        size=(n_samples, length),
        dtype=np.int64
    )

    delta = np.array([
        0,
        1,
        2,
        -1,
        -2
    ])

    state = np.zeros(
        n_samples,
        dtype=np.int64
    )

    for t in range(length):
        state = (
            state + delta[operations[:, t]]
        ) % 5

    return operations, state


print("Task generators created.")
print()
print("Parity:")
print("  Input  = operation tokens")
print("  Target = final parity state")
print()
print("Five-state:")
print("  Input  = operation tokens")
print("  Target = final state")
print()
print("Correct states are never provided as inputs.")

Task generators created.

Parity:
  Input  = operation tokens
  Target = final parity state

Five-state:
  Input  = operation tokens
  Target = final state

Correct states are never provided as inputs.


In [4]:
# ============================================================
# RLT REIMPLEMENTATION — STEP 4
# Task Sanity Check
# ============================================================

# ------------------------------------------------------------
# PARITY
# ------------------------------------------------------------

parity_x, parity_y = generate_parity(
    n_samples=5,
    length=8,
    seed=42
)

print("PARITY TASK")
print("-----------")

for i in range(5):
    sequence = parity_x[i]
    target = parity_y[i]

    manual = 0
    for op in sequence:
        manual ^= op

    print(
        f"Sequence: {sequence} | "
        f"Target: {target} | "
        f"Manual: {manual} | "
        f"Match: {target == manual}"
    )


print()


# ------------------------------------------------------------
# FIVE-STATE
# ------------------------------------------------------------

five_x, five_y = generate_five_state(
    n_samples=5,
    length=8,
    seed=42
)

print("FIVE-STATE TASK")
print("---------------")

delta = [0, 1, 2, -1, -2]

for i in range(5):
    sequence = five_x[i]
    target = five_y[i]

    manual = 0

    for op in sequence:
        manual = (manual + delta[op]) % 5

    print(
        f"Sequence: {sequence} | "
        f"Target: {target} | "
        f"Manual: {manual} | "
        f"Match: {target == manual}"
    )

PARITY TASK
-----------
Sequence: [0 1 1 0 0 1 0 1] | Target: 0 | Manual: 0 | Match: True
Sequence: [0 0 1 1 1 1 1 1] | Target: 0 | Manual: 0 | Match: True
Sequence: [1 0 1 0 1 0 0 1] | Target: 0 | Manual: 0 | Match: True
Sequence: [1 1 0 1 1 0 0 0] | Target: 0 | Manual: 0 | Match: True
Sequence: [0 1 1 0 1 1 0 1] | Target: 1 | Manual: 1 | Match: True

FIVE-STATE TASK
---------------
Sequence: [0 3 3 2 2 4 0 3] | Target: 4 | Manual: 4 | Match: True
Sequence: [1 0 2 4 3 3 3 3] | Target: 2 | Manual: 2 | Match: True
Sequence: [2 0 4 2 2 1 0 4] | Target: 3 | Manual: 3 | Match: True
Sequence: [3 3 2 4 2 2 2 1] | Target: 0 | Manual: 0 | Match: True
Sequence: [0 2 4 0 4 4 1 3] | Target: 1 | Manual: 1 | Match: True


In [5]:
# ============================================================
# RLT REIMPLEMENTATION — STEP 5
# RMSNorm
# ============================================================

class RMSNorm(nn.Module):
    def __init__(self, d_model, eps=1e-8):
        super().__init__()

        self.eps = eps

        self.weight = nn.Parameter(
            torch.ones(d_model)
        )

    def forward(self, x):

        # Root mean square
        rms = torch.sqrt(
            x.pow(2).mean(
                dim=-1,
                keepdim=True
            ) + self.eps
        )

        # Normalize and scale
        return (
            x / rms
        ) * self.weight


# ------------------------------------------------------------
# Sanity check
# ------------------------------------------------------------

test = torch.randn(
    4, 8, 32,
    device=device
)

norm = RMSNorm(32).to(device)

output = norm(test)

print("Input shape :", test.shape)
print("Output shape:", output.shape)
print(
    "Output finite:",
    torch.isfinite(output).all().item()
)

print("RMSNorm ready.")

Input shape : torch.Size([4, 8, 32])
Output shape: torch.Size([4, 8, 32])
Output finite: True
RMSNorm ready.


In [7]:
class SlidingWindowAttentionV2(nn.Module):
    def __init__(self, d_model, n_heads=4, window_size=8):
        super().__init__()

        assert d_model % n_heads == 0

        self.d_model = d_model
        self.n_heads = n_heads
        self.window_size = window_size
        self.head_dim = d_model // n_heads

        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)

        self.out_proj = nn.Linear(d_model, d_model, bias=False)

        self.norm = RMSNorm(d_model)

    def forward(self, x, cache=None):

        batch_size = x.size(0)

        # ------------------------------------------------
        # 1. Project current token
        # ------------------------------------------------
        q = self.q_proj(x)
        k_new = self.k_proj(x)
        v_new = self.v_proj(x)

        # ------------------------------------------------
        # 2. Update KV cache
        # Cache remains 3D:
        # [batch, sequence, d_model]
        # ------------------------------------------------
        if cache is None:
            k = k_new
            v = v_new
        else:
            k_old, v_old = cache

            k = torch.cat([k_old, k_new], dim=1)
            v = torch.cat([v_old, v_new], dim=1)

        # ------------------------------------------------
        # 3. Keep only the latest window
        # ------------------------------------------------
        k = k[:, -self.window_size:, :]
        v = v[:, -self.window_size:, :]

        # ------------------------------------------------
        # 4. Convert to multi-head format
        # ------------------------------------------------
        q = q.view(
            batch_size,
            1,
            self.n_heads,
            self.head_dim
        ).transpose(1, 2)

        k_heads = k.view(
            batch_size,
            -1,
            self.n_heads,
            self.head_dim
        ).transpose(1, 2)

        v_heads = v.view(
            batch_size,
            -1,
            self.n_heads,
            self.head_dim
        ).transpose(1, 2)

        # ------------------------------------------------
        # 5. Attention
        # ------------------------------------------------
        scores = torch.matmul(
            q,
            k_heads.transpose(-2, -1)
        ) / math.sqrt(self.head_dim)

        attention = F.softmax(scores, dim=-1)

        output = torch.matmul(
            attention,
            v_heads
        )

        # ------------------------------------------------
        # 6. Merge heads
        # ------------------------------------------------
        output = output.transpose(1, 2).contiguous()

        output = output.view(
            batch_size,
            1,
            self.d_model
        )

        # ------------------------------------------------
        # 7. Output projection + residual + RMSNorm
        # ------------------------------------------------
        output = self.out_proj(output)

        output = self.norm(x + output)

        # Return 3D cache
        return output, (k, v)


# --------------------------------------------------------
# Sanity check
# --------------------------------------------------------

attention = SlidingWindowAttentionV2(
    d_model=32,
    n_heads=4,
    window_size=8
).to(device)

x = torch.randn(4, 1, 32, device=device)

cache = None

for step in range(12):

    output, cache = attention(x, cache)

    if step in [0, 7, 11]:
        print(
            f"Step {step+1:2d} | "
            f"Output: {output.shape} | "
            f"Cache: {cache[0].shape}"
        )

print("Sliding-window attention ready.")

Step  1 | Output: torch.Size([4, 1, 32]) | Cache: torch.Size([4, 1, 32])
Step  8 | Output: torch.Size([4, 1, 32]) | Cache: torch.Size([4, 8, 32])
Step 12 | Output: torch.Size([4, 1, 32]) | Cache: torch.Size([4, 8, 32])
Sliding-window attention ready.


In [8]:
class RecurrentStateUpdate(nn.Module):
    def __init__(self, d_model):
        super().__init__()

        self.gru = nn.GRUCell(
            input_size=d_model,
            hidden_size=d_model
        )

    def forward(self, x, state=None):

        batch_size = x.size(0)

        # Initial recurrent state
        if state is None:
            state = torch.zeros(
                batch_size,
                x.size(-1),
                device=x.device,
                dtype=x.dtype
            )

        # Recurrent update
        new_state = self.gru(
            x,
            state
        )

        return new_state


# --------------------------------------------------------
# Sanity check
# --------------------------------------------------------

recurrent = RecurrentStateUpdate(
    d_model=32
).to(device)

state = None

for step in range(5):

    x = torch.randn(
        4,
        32,
        device=device
    )

    state = recurrent(x, state)

    print(
        f"Step {step+1} | State: {state.shape}"
    )

print("Recurrent state update ready.")

Step 1 | State: torch.Size([4, 32])
Step 2 | State: torch.Size([4, 32])
Step 3 | State: torch.Size([4, 32])
Step 4 | State: torch.Size([4, 32])
Step 5 | State: torch.Size([4, 32])
Recurrent state update ready.


In [9]:
#8
class StateTokenMerge(nn.Module):
    def __init__(self, d_model):
        super().__init__()

        self.merge = nn.Linear(
            2 * d_model,
            d_model
        )

        self.norm = RMSNorm(d_model)

    def forward(self, encoder_state, previous_state):

        combined = torch.cat(
            [encoder_state, previous_state],
            dim=-1
        )

        merged = self.merge(combined)

        merged = self.norm(merged)

        return merged


# --------------------------------------------------------
# Sanity check
# --------------------------------------------------------

merger = StateTokenMerge(
    d_model=32
).to(device)

encoder_state = torch.randn(
    4, 32,
    device=device
)

previous_state = torch.randn(
    4, 32,
    device=device
)

merged = merger(
    encoder_state,
    previous_state
)

print("Encoder state :", encoder_state.shape)
print("Previous state:", previous_state.shape)
print("Merged state  :", merged.shape)

print("State-token merge ready.")

Encoder state : torch.Size([4, 32])
Previous state: torch.Size([4, 32])
Merged state  : torch.Size([4, 32])
State-token merge ready.


In [10]:
#9
class TinyRLTDecoderBlock(nn.Module):
    def __init__(
        self,
        d_model=32,
        n_heads=4,
        window_size=8
    ):
        super().__init__()

        self.merge = StateTokenMerge(d_model)

        self.attention = SlidingWindowAttentionV2(
            d_model=d_model,
            n_heads=n_heads,
            window_size=window_size
        )

        self.recurrent = RecurrentStateUpdate(
            d_model
        )

    def forward(
        self,
        encoder_state,
        previous_state=None,
        cache=None
    ):

        batch_size = encoder_state.size(0)

        # Initial recurrent state
        if previous_state is None:
            previous_state = torch.zeros(
                batch_size,
                encoder_state.size(-1),
                device=encoder_state.device,
                dtype=encoder_state.dtype
            )

        # ------------------------------------------------
        # 1. Merge encoder representation + previous state
        # ------------------------------------------------
        merged = self.merge(
            encoder_state,
            previous_state
        )

        # ------------------------------------------------
        # 2. Sliding-window attention
        # ------------------------------------------------
        attention_input = merged.unsqueeze(1)

        attention_output, new_cache = self.attention(
            attention_input,
            cache
        )

        attention_output = attention_output.squeeze(1)

        # ------------------------------------------------
        # 3. Recurrent state update
        # ------------------------------------------------
        new_state = self.recurrent(
            attention_output,
            previous_state
        )

        return new_state, new_cache


# --------------------------------------------------------
# Sanity check
# --------------------------------------------------------

decoder = TinyRLTDecoderBlock(
    d_model=32,
    n_heads=4,
    window_size=8
).to(device)

encoder_state = torch.randn(
    4,
    32,
    device=device
)

state = None
cache = None

for step in range(12):

    state, cache = decoder(
        encoder_state,
        state,
        cache
    )

    if step in [0, 7, 11]:
        print(
            f"Step {step+1:2d} | "
            f"State: {state.shape} | "
            f"Cache: {cache[0].shape}"
        )

print("Tiny RLT decoder block ready.")

Step  1 | State: torch.Size([4, 32]) | Cache: torch.Size([4, 1, 32])
Step  8 | State: torch.Size([4, 32]) | Cache: torch.Size([4, 8, 32])
Step 12 | State: torch.Size([4, 32]) | Cache: torch.Size([4, 8, 32])
Tiny RLT decoder block ready.


In [11]:
#10
class TinyCausalEncoder(nn.Module):
    def __init__(
        self,
        d_model=32,
        n_heads=4,
        ff_dim=64
    ):
        super().__init__()

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=ff_dim,
            dropout=0.0,
            activation="gelu",
            batch_first=True,
            norm_first=False
        )

        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=1
        )

        self.norm = RMSNorm(d_model)

    def forward(self, x):

        seq_len = x.size(1)

        # Causal mask:
        # token t cannot see future tokens
        causal_mask = torch.triu(
            torch.ones(
                seq_len,
                seq_len,
                device=x.device,
                dtype=torch.bool
            ),
            diagonal=1
        )

        output = self.encoder(
            x,
            mask=causal_mask
        )

        output = self.norm(output)

        return output


# --------------------------------------------------------
# Sanity check
# --------------------------------------------------------

encoder = TinyCausalEncoder(
    d_model=32,
    n_heads=4,
    ff_dim=64
).to(device)

x = torch.randn(
    4,
    32,
    32,
    device=device
)

encoder_output = encoder(x)

print("Input :", x.shape)
print("Output:", encoder_output.shape)
print(
    "Finite:",
    torch.isfinite(encoder_output).all().item()
)

print("Causal encoder ready.")

Input : torch.Size([4, 32, 32])
Output: torch.Size([4, 32, 32])
Finite: True
Causal encoder ready.


In [12]:
#11
class OperationEmbedding(nn.Module):
    def __init__(self, num_operations, d_model=32):
        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings=num_operations,
            embedding_dim=d_model
        )

    def forward(self, operations):
        return self.embedding(operations)


# --------------------------------------------------------
# Sanity check: Parity
# --------------------------------------------------------

parity_embedding = OperationEmbedding(
    num_operations=2,
    d_model=32
).to(device)

operations = torch.randint(
    0,
    2,
    (4, 32),
    device=device
)

embedded = parity_embedding(operations)

print("Operation IDs:", operations.shape)
print("Embedded     :", embedded.shape)
print("Finite       :", torch.isfinite(embedded).all().item())

# --------------------------------------------------------
# Sanity check: Five-state
# --------------------------------------------------------

five_state_embedding = OperationEmbedding(
    num_operations=5,
    d_model=32
).to(device)

operations_5 = torch.randint(
    0,
    5,
    (4, 32),
    device=device
)

embedded_5 = five_state_embedding(operations_5)

print("Five-state IDs:", operations_5.shape)
print("Five-state emb:", embedded_5.shape)

print("Operation embedding ready.")

Operation IDs: torch.Size([4, 32])
Embedded     : torch.Size([4, 32, 32])
Finite       : True
Five-state IDs: torch.Size([4, 32])
Five-state emb: torch.Size([4, 32, 32])
Operation embedding ready.


In [13]:
#12
class TokenPositionEmbedding(nn.Module):
    def __init__(
        self,
        max_length=128,
        d_model=32
    ):
        super().__init__()

        self.position_embedding = nn.Embedding(
            max_length,
            d_model
        )

    def forward(self, x):

        batch_size, seq_len, d_model = x.shape

        positions = torch.arange(
            seq_len,
            device=x.device
        )

        positions = positions.unsqueeze(0).expand(
            batch_size,
            seq_len
        )

        position_vectors = self.position_embedding(
            positions
        )

        return x + position_vectors


# --------------------------------------------------------
# Sanity check
# --------------------------------------------------------

position_embedding = TokenPositionEmbedding(
    max_length=128,
    d_model=32
).to(device)

x = torch.randn(
    4,
    32,
    32,
    device=device
)

output = position_embedding(x)

print("Input :", x.shape)
print("Output:", output.shape)
print(
    "Finite:",
    torch.isfinite(output).all().item()
)

print("Token-position embedding ready.")

Input : torch.Size([4, 32, 32])
Output: torch.Size([4, 32, 32])
Finite: True
Token-position embedding ready.


In [14]:
#13
class TinyRLT(nn.Module):
    def __init__(
        self,
        num_operations,
        num_states,
        d_model=32,
        n_heads=4,
        ff_dim=64,
        window_size=8,
        max_length=128
    ):
        super().__init__()

        self.d_model = d_model

        # ------------------------------------------------
        # Input representation
        # ------------------------------------------------
        self.operation_embedding = OperationEmbedding(
            num_operations=num_operations,
            d_model=d_model
        )

        self.position_embedding = TokenPositionEmbedding(
            max_length=max_length,
            d_model=d_model
        )

        # ------------------------------------------------
        # Causal encoder
        # ------------------------------------------------
        self.encoder = TinyCausalEncoder(
            d_model=d_model,
            n_heads=n_heads,
            ff_dim=ff_dim
        )

        # ------------------------------------------------
        # Recurrent decoder
        # ------------------------------------------------
        self.decoder = TinyRLTDecoderBlock(
            d_model=d_model,
            n_heads=n_heads,
            window_size=window_size
        )

        # ------------------------------------------------
        # Output head
        # ------------------------------------------------
        self.output_norm = RMSNorm(d_model)

        self.output_head = nn.Linear(
            d_model,
            num_states
        )

    def forward(self, operations):

        batch_size, seq_len = operations.shape

        # ------------------------------------------------
        # 1. Operation embedding
        # ------------------------------------------------
        x = self.operation_embedding(operations)

        # ------------------------------------------------
        # 2. Positional information
        # ------------------------------------------------
        x = self.position_embedding(x)

        # ------------------------------------------------
        # 3. Causal encoder
        # ------------------------------------------------
        encoder_outputs = self.encoder(x)

        # ------------------------------------------------
        # 4. Recurrent decoding
        # ------------------------------------------------
        state = None
        cache = None

        for t in range(seq_len):

            encoder_state = encoder_outputs[:, t, :]

            state, cache = self.decoder(
                encoder_state,
                state,
                cache
            )

        # ------------------------------------------------
        # 5. Predict final state
        # ------------------------------------------------
        state = self.output_norm(state)

        logits = self.output_head(state)

        return logits


# --------------------------------------------------------
# Parity model sanity check
# --------------------------------------------------------

model = TinyRLT(
    num_operations=2,
    num_states=2,
    d_model=32,
    n_heads=4,
    ff_dim=64,
    window_size=8,
    max_length=128
).to(device)

operations = torch.randint(
    0,
    2,
    (4, 32),
    device=device
)

logits = model(operations)

print("Input :", operations.shape)
print("Logits:", logits.shape)
print(
    "Finite:",
    torch.isfinite(logits).all().item()
)

print("Tiny RLT assembled successfully.")

Input : torch.Size([4, 32])
Logits: torch.Size([4, 2])
Finite: True
Tiny RLT assembled successfully.


In [15]:
#14
def count_parameters(model):
    return sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )


total_params = count_parameters(model)

print(
    f"Total trainable parameters: {total_params:,}"
)

print(
    f"Target from experiment:     {TARGET_PARAMS:,}"
)

print(
    f"Difference:                  "
    f"{total_params - TARGET_PARAMS:+,}"
)

print(
    f"Relative to target:          "
    f"{100 * total_params / TARGET_PARAMS:.1f}%"
)

Total trainable parameters: 25,410
Target from experiment:     79,000
Difference:                  -53,590
Relative to target:          32.2%


In [16]:
model_79k_test = TinyRLT(
    num_operations=2,
    num_states=2,
    d_model=32,
    n_heads=4,
    ff_dim=256,
    window_size=8,
    max_length=128
).to(device)

params = count_parameters(model_79k_test)

print(f"Parameters with ff_dim=256: {params:,}")
print(f"Target:                    {TARGET_PARAMS:,}")
print(f"Difference:                {params - TARGET_PARAMS:+,}")

Parameters with ff_dim=256: 37,890
Target:                    79,000
Difference:                -41,110


In [20]:
#16
print("Parameter breakdown")
print("-" * 45)

for name, module in model_79k_test.named_children():
    params = count_parameters(module)
    print(f"{name:25s}: {params:8,}")

print("-" * 45)
print(f"{'TOTAL':25s}: {count_parameters(model_79k_test):8,}")

Parameter breakdown
---------------------------------------------
operation_embedding      :       64
position_embedding       :    4,096
encoder                  :   21,056
decoder                  :   12,576
output_norm              :       32
output_head              :       66
---------------------------------------------
TOTAL                    :   37,890


In [21]:
print(model_79k_test)

TinyRLT(
  (operation_embedding): OperationEmbedding(
    (embedding): Embedding(2, 32)
  )
  (position_embedding): TokenPositionEmbedding(
    (position_embedding): Embedding(128, 32)
  )
  (encoder): TinyCausalEncoder(
    (encoder): TransformerEncoder(
      (layers): ModuleList(
        (0): TransformerEncoderLayer(
          (self_attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=32, out_features=32, bias=True)
          )
          (linear1): Linear(in_features=32, out_features=256, bias=True)
          (dropout): Dropout(p=0.0, inplace=False)
          (linear2): Linear(in_features=256, out_features=32, bias=True)
          (norm1): LayerNorm((32,), eps=1e-05, elementwise_affine=True)
          (norm2): LayerNorm((32,), eps=1e-05, elementwise_affine=True)
          (dropout1): Dropout(p=0.0, inplace=False)
          (dropout2): Dropout(p=0.0, inplace=False)
        )
      )
    )
    (norm): RMSNorm()
  )
  (decoder): TinyRLTDecoderB

In [22]:
for name, param in model_79k_test.named_parameters():
    print(f"{name:65s} {tuple(param.shape)}")

operation_embedding.embedding.weight                              (2, 32)
position_embedding.position_embedding.weight                      (128, 32)
encoder.encoder.layers.0.self_attn.in_proj_weight                 (96, 32)
encoder.encoder.layers.0.self_attn.in_proj_bias                   (96,)
encoder.encoder.layers.0.self_attn.out_proj.weight                (32, 32)
encoder.encoder.layers.0.self_attn.out_proj.bias                  (32,)
encoder.encoder.layers.0.linear1.weight                           (256, 32)
encoder.encoder.layers.0.linear1.bias                             (256,)
encoder.encoder.layers.0.linear2.weight                           (32, 256)
encoder.encoder.layers.0.linear2.bias                             (32,)
encoder.encoder.layers.0.norm1.weight                             (32,)
encoder.encoder.layers.0.norm1.bias                               (32,)
encoder.encoder.layers.0.norm2.weight                             (32,)
encoder.encoder.layers.0.norm2.bias        

In [23]:
decoder_parameter_ids = [
    id(param)
    for param in model_79k_test.decoder.parameters()
]

print("Decoder parameter tensors:", len(decoder_parameter_ids))
print(
    "Unique decoder tensors:    ",
    len(set(decoder_parameter_ids))
)

print(
    "Weights reused across steps:",
    len(decoder_parameter_ids) == len(set(decoder_parameter_ids))
)

Decoder parameter tensors: 12
Unique decoder tensors:     12
Weights reused across steps: True


In [24]:
model_79k_test.train()

# Small test batch
operations = torch.randint(
    0,
    2,
    (4, 32),
    device=device
)

targets = torch.randint(
    0,
    2,
    (4,),
    device=device
)

# Forward pass
logits = model_79k_test(operations)

# Loss
loss = F.cross_entropy(
    logits,
    targets
)

# Backward pass
model_79k_test.zero_grad()
loss.backward()

print("Loss:", loss.item())

# Check gradients
gradient_info = []

for name, param in model_79k_test.named_parameters():

    if param.requires_grad:

        has_gradient = param.grad is not None

        if has_gradient:
            grad_norm = param.grad.norm().item()
        else:
            grad_norm = 0.0

        gradient_info.append(
            (name, has_gradient, grad_norm)
        )

missing = [
    name
    for name, has_grad, _ in gradient_info
    if not has_grad
]

nonzero = [
    name
    for name, has_grad, grad_norm in gradient_info
    if has_grad and grad_norm > 0
]

print("Trainable tensors:", len(gradient_info))
print("Missing gradients:", len(missing))
print("Non-zero gradients:", len(nonzero))

if missing:
    print("\nMissing:")
    for name in missing:
        print(" ", name)

print("\nGradient flow check complete.")

Loss: 0.7868146896362305
Trainable tensors: 30
Missing gradients: 0
Non-zero gradients: 30

Gradient flow check complete.


In [25]:
# --------------------------------------------------------
# Tiny overfit dataset
# --------------------------------------------------------

X_tiny, y_tiny = generate_parity(
    n_samples=32,
    length=32,
    seed=123
)

X_tiny = torch.tensor(
    X_tiny,
    dtype=torch.long,
    device=device
)

y_tiny = torch.tensor(
    y_tiny,
    dtype=torch.long,
    device=device
)

# Fresh model
tiny_model = TinyRLT(
    num_operations=2,
    num_states=2,
    d_model=32,
    n_heads=4,
    ff_dim=256,
    window_size=8,
    max_length=128
).to(device)

optimizer = torch.optim.AdamW(
    tiny_model.parameters(),
    lr=3e-4
)

tiny_model.train()

for step in range(501):

    optimizer.zero_grad()

    logits = tiny_model(X_tiny)

    loss = F.cross_entropy(
        logits,
        y_tiny
    )

    loss.backward()

    optimizer.step()

    if step % 100 == 0:

        predictions = logits.argmax(dim=-1)

        accuracy = (
            predictions == y_tiny
        ).float().mean().item() * 100

        print(
            f"Step {step:3d} | "
            f"Loss {loss.item():.4f} | "
            f"Accuracy {accuracy:.2f}%"
        )

Step   0 | Loss 0.6079 | Accuracy 68.75%
Step 100 | Loss 0.0101 | Accuracy 100.00%
Step 200 | Loss 0.0039 | Accuracy 100.00%
Step 300 | Loss 0.0023 | Accuracy 100.00%
Step 400 | Loss 0.0016 | Accuracy 100.00%
Step 500 | Loss 0.0012 | Accuracy 100.00%


In [26]:
# --------------------------------------------------------
# Full parity training dataset
# --------------------------------------------------------

N_TRAIN = 10_000

X_train, y_train = generate_parity(
    n_samples=N_TRAIN,
    length=TRAIN_LENGTH,
    seed=SEEDS[0]
)

X_train = torch.tensor(
    X_train,
    dtype=torch.long,
    device=device
)

y_train = torch.tensor(
    y_train,
    dtype=torch.long,
    device=device
)

print("Training inputs :", X_train.shape)
print("Training targets:", y_train.shape)

print(
    "Class 0:",
    (y_train == 0).sum().item()
)

print(
    "Class 1:",
    (y_train == 1).sum().item()
)

print("Dataset ready.")

Training inputs : torch.Size([10000, 32])
Training targets: torch.Size([10000])
Class 0: 4999
Class 1: 5001
Dataset ready.


In [27]:
# --------------------------------------------------------
# Full parity training
# --------------------------------------------------------

BATCH_SIZE = 64
TRAIN_STEPS = 2000
LR = 3e-4

train_model = TinyRLT(
    num_operations=2,
    num_states=2,
    d_model=32,
    n_heads=4,
    ff_dim=256,
    window_size=8,
    max_length=128
).to(device)

optimizer = torch.optim.AdamW(
    train_model.parameters(),
    lr=LR
)

train_model.train()

N = X_train.size(0)

for step in range(1, TRAIN_STEPS + 1):

    # Random mini-batch
    indices = torch.randint(
        0,
        N,
        (BATCH_SIZE,),
        device=device
    )

    batch_x = X_train[indices]
    batch_y = y_train[indices]

    # Forward
    logits = train_model(batch_x)

    # Loss
    loss = F.cross_entropy(
        logits,
        batch_y
    )

    # Backpropagation
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    # Progress
    if step == 1 or step % 200 == 0:

        predictions = logits.argmax(dim=-1)

        accuracy = (
            predictions == batch_y
        ).float().mean().item() * 100

        print(
            f"Step {step:4d} | "
            f"Loss {loss.item():.4f} | "
            f"Batch Accuracy {accuracy:.2f}%"
        )

print("Full parity training complete.")

Step    1 | Loss 0.7921 | Batch Accuracy 42.19%
Step  200 | Loss 0.6894 | Batch Accuracy 60.94%
Step  400 | Loss 0.7038 | Batch Accuracy 48.44%
Step  600 | Loss 0.6971 | Batch Accuracy 40.62%
Step  800 | Loss 0.6933 | Batch Accuracy 48.44%
Step 1000 | Loss 0.6979 | Batch Accuracy 45.31%
Step 1200 | Loss 0.6970 | Batch Accuracy 43.75%
Step 1400 | Loss 0.6927 | Batch Accuracy 50.00%
Step 1600 | Loss 0.6922 | Batch Accuracy 54.69%
Step 1800 | Loss 0.6899 | Batch Accuracy 53.12%
Step 2000 | Loss 0.6929 | Batch Accuracy 50.00%
Full parity training complete.


In [28]:
train_model.eval()

with torch.no_grad():
    train_logits = train_model(X_train)
    train_predictions = train_logits.argmax(dim=-1)

    train_accuracy = (
        train_predictions == y_train
    ).float().mean().item() * 100

    train_loss = F.cross_entropy(
        train_logits,
        y_train
    ).item()

print(f"Training Loss:     {train_loss:.4f}")
print(f"Training Accuracy: {train_accuracy:.2f}%")

print("Training-set evaluation complete.")

Training Loss:     0.6935
Training Accuracy: 50.01%
Training-set evaluation complete.


In [29]:
class SimpleGRU(nn.Module):
    def __init__(self, num_operations=2, d_model=32, num_states=2):
        super().__init__()

        self.embedding = nn.Embedding(num_operations, d_model)
        self.gru = nn.GRU(
            input_size=d_model,
            hidden_size=d_model,
            batch_first=True
        )
        self.head = nn.Linear(d_model, num_states)

    def forward(self, operations):
        x = self.embedding(operations)
        output, hidden = self.gru(x)
        final_state = hidden[-1]
        return self.head(final_state)


# Fresh GRU
gru_test = SimpleGRU().to(device)

optimizer = torch.optim.AdamW(
    gru_test.parameters(),
    lr=3e-4
)

print("Parameters:", sum(p.numel() for p in gru_test.parameters()))

# Train for 500 updates
gru_test.train()

for step in range(501):
    idx = torch.randint(
        0,
        len(X_train),
        (64,),
        device=device
    )

    batch_x = X_train[idx]
    batch_y = y_train[idx]

    logits = gru_test(batch_x)
    loss = F.cross_entropy(logits, batch_y)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % 100 == 0:
        accuracy = (
            logits.argmax(dim=-1) == batch_y
        ).float().mean().item() * 100

        print(
            f"Step {step:3d} | "
            f"Loss {loss.item():.4f} | "
            f"Accuracy {accuracy:.2f}%"
        )

print("GRU parity test complete.")

Parameters: 6466
Step   0 | Loss 0.7076 | Accuracy 54.69%
Step 100 | Loss 0.6982 | Accuracy 39.06%
Step 200 | Loss 0.6932 | Accuracy 45.31%
Step 300 | Loss 0.6898 | Accuracy 56.25%
Step 400 | Loss 0.6891 | Accuracy 59.38%
Step 500 | Loss 0.6970 | Accuracy 37.50%
GRU parity test complete.


In [30]:
# Inspect some training examples manually

for i in range(10):
    operations = X_train[i].detach().cpu().numpy()
    true_label = y_train[i].item()

    # Calculate parity directly from the sequence
    calculated_label = int(operations.sum() % 2)

    print(
        f"Example {i}: "
        f"Sum = {operations.sum():2d} | "
        f"Label = {true_label} | "
        f"Calculated = {calculated_label} | "
        f"Match = {true_label == calculated_label}"
    )

print("\nClass distribution:")
print("Class 0:", (y_train == 0).sum().item())
print("Class 1:", (y_train == 1).sum().item())

Example 0: Sum = 18 | Label = 0 | Calculated = 0 | Match = True
Example 1: Sum = 18 | Label = 0 | Calculated = 0 | Match = True
Example 2: Sum = 14 | Label = 0 | Calculated = 0 | Match = True
Example 3: Sum = 18 | Label = 0 | Calculated = 0 | Match = True
Example 4: Sum = 17 | Label = 1 | Calculated = 1 | Match = True
Example 5: Sum = 13 | Label = 1 | Calculated = 1 | Match = True
Example 6: Sum = 13 | Label = 1 | Calculated = 1 | Match = True
Example 7: Sum = 15 | Label = 1 | Calculated = 1 | Match = True
Example 8: Sum = 13 | Label = 1 | Calculated = 1 | Match = True
Example 9: Sum = 17 | Label = 1 | Calculated = 1 | Match = True

Class distribution:
Class 0: 4999
Class 1: 5001


In [31]:
# Use only 32 examples
X_tiny = X_train[:32]
y_tiny = y_train[:32]

tiny_gru = SimpleGRU().to(device)

optimizer = torch.optim.AdamW(
    tiny_gru.parameters(),
    lr=3e-4
)

tiny_gru.train()

for step in range(501):
    logits = tiny_gru(X_tiny)
    loss = F.cross_entropy(logits, y_tiny)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % 100 == 0:
        accuracy = (
            logits.argmax(dim=-1) == y_tiny
        ).float().mean().item() * 100

        print(
            f"Step {step:3d} | "
            f"Loss {loss.item():.4f} | "
            f"Accuracy {accuracy:.2f}%"
        )

print("GRU tiny overfit test complete.")

Step   0 | Loss 0.7315 | Accuracy 43.75%
Step 100 | Loss 0.6242 | Accuracy 62.50%
Step 200 | Loss 0.4561 | Accuracy 78.12%
Step 300 | Loss 0.2824 | Accuracy 87.50%
Step 400 | Loss 0.0930 | Accuracy 100.00%
Step 500 | Loss 0.0298 | Accuracy 100.00%
GRU tiny overfit test complete.


In [32]:
gru_full = SimpleGRU().to(device)

optimizer = torch.optim.AdamW(
    gru_full.parameters(),
    lr=1e-3
)

gru_full.train()

for step in range(2001):
    idx = torch.randint(
        0,
        len(X_train),
        (64,),
        device=device
    )

    batch_x = X_train[idx]
    batch_y = y_train[idx]

    logits = gru_full(batch_x)
    loss = F.cross_entropy(logits, batch_y)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % 200 == 0:
        accuracy = (
            logits.argmax(dim=-1) == batch_y
        ).float().mean().item() * 100

        print(
            f"Step {step:4d} | "
            f"Loss {loss.item():.4f} | "
            f"Accuracy {accuracy:.2f}%"
        )

print("Full GRU training complete.")

Step    0 | Loss 0.7261 | Accuracy 50.00%
Step  200 | Loss 0.7025 | Accuracy 46.88%
Step  400 | Loss 0.6964 | Accuracy 40.62%
Step  600 | Loss 0.6925 | Accuracy 51.56%
Step  800 | Loss 0.6948 | Accuracy 48.44%
Step 1000 | Loss 0.6922 | Accuracy 53.12%
Step 1200 | Loss 0.6919 | Accuracy 54.69%
Step 1400 | Loss 0.6903 | Accuracy 57.81%
Step 1600 | Loss 0.6916 | Accuracy 59.38%
Step 1800 | Loss 0.6864 | Accuracy 57.81%
Step 2000 | Loss 0.7053 | Accuracy 42.19%
Full GRU training complete.


In [33]:
# Generate a shorter parity dataset
X_short_np, y_short_np = generate_parity(
    n_samples=10_000,
    length=8,
    seed=42
)

X_short = torch.tensor(
    X_short_np,
    dtype=torch.long,
    device=device
)

y_short = torch.tensor(
    y_short_np,
    dtype=torch.long,
    device=device
)

short_gru = SimpleGRU().to(device)

optimizer = torch.optim.AdamW(
    short_gru.parameters(),
    lr=1e-3
)

short_gru.train()

for step in range(2001):
    idx = torch.randint(
        0,
        len(X_short),
        (64,),
        device=device
    )

    batch_x = X_short[idx]
    batch_y = y_short[idx]

    logits = short_gru(batch_x)
    loss = F.cross_entropy(logits, batch_y)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % 200 == 0:
        accuracy = (
            logits.argmax(dim=-1) == batch_y
        ).float().mean().item() * 100

        print(
            f"Step {step:4d} | "
            f"Loss {loss.item():.4f} | "
            f"Accuracy {accuracy:.2f}%"
        )

print("Short-sequence GRU test complete.")

Step    0 | Loss 0.7276 | Accuracy 50.00%
Step  200 | Loss 0.6909 | Accuracy 51.56%
Step  400 | Loss 0.6929 | Accuracy 51.56%
Step  600 | Loss 0.6944 | Accuracy 46.88%
Step  800 | Loss 0.6900 | Accuracy 53.12%
Step 1000 | Loss 0.6942 | Accuracy 50.00%
Step 1200 | Loss 0.6872 | Accuracy 56.25%
Step 1400 | Loss 0.6941 | Accuracy 45.31%
Step 1600 | Loss 0.6935 | Accuracy 50.00%
Step 1800 | Loss 0.6977 | Accuracy 40.62%
Step 2000 | Loss 0.6937 | Accuracy 51.56%
Short-sequence GRU test complete.


In [34]:
# --------------------------------------------------------
# Step 30 — Simple parity sanity check
# --------------------------------------------------------

class ParitySumModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(1, 32),
            nn.ReLU(),
            nn.Linear(32, 2)
        )

    def forward(self, x):
        # Sum the binary operation tokens
        s = x.float().sum(dim=1, keepdim=True)
        return self.net(s)


sum_model = ParitySumModel().to(device)

optimizer = torch.optim.AdamW(
    sum_model.parameters(),
    lr=1e-3
)

for step in range(1001):

    idx = torch.randint(
        0,
        len(X_train),
        (128,),
        device=device
    )

    batch_x = X_train[idx]
    batch_y = y_train[idx]

    logits = sum_model(batch_x)
    loss = F.cross_entropy(logits, batch_y)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % 200 == 0:

        accuracy = (
            logits.argmax(dim=-1) == batch_y
        ).float().mean().item() * 100

        print(
            f"Step {step:4d} | "
            f"Loss {loss.item():.4f} | "
            f"Accuracy {accuracy:.2f}%"
        )

print("Parity sum sanity check complete.")

Step    0 | Loss 0.7689 | Accuracy 51.56%
Step  200 | Loss 0.6909 | Accuracy 55.47%
Step  400 | Loss 0.6969 | Accuracy 46.09%
Step  600 | Loss 0.7002 | Accuracy 46.09%
Step  800 | Loss 0.7018 | Accuracy 51.56%
Step 1000 | Loss 0.6906 | Accuracy 53.91%
Parity sum sanity check complete.


In [35]:
pred = X_train.sum(dim=1) % 2

print("Pred dtype:", pred.dtype)
print("Target dtype:", y_train.dtype)

print("Correct predictions:",
      (pred == y_train).sum().item(),
      "/", len(y_train))

print("Accuracy:",
      (pred == y_train).float().mean().item() * 100)

print("First 20:")
for i in range(20):
    print(
        "sum =", X_train[i].sum().item(),
        "| predicted =", pred[i].item(),
        "| target =", y_train[i].item()
    )

Pred dtype: torch.int64
Target dtype: torch.int64
Correct predictions: 10000 / 10000
Accuracy: 100.0
First 20:
sum = 18 | predicted = 0 | target = 0
sum = 18 | predicted = 0 | target = 0
sum = 14 | predicted = 0 | target = 0
sum = 18 | predicted = 0 | target = 0
sum = 17 | predicted = 1 | target = 1
sum = 13 | predicted = 1 | target = 1
sum = 13 | predicted = 1 | target = 1
sum = 15 | predicted = 1 | target = 1
sum = 13 | predicted = 1 | target = 1
sum = 17 | predicted = 1 | target = 1
sum = 16 | predicted = 0 | target = 0
sum = 15 | predicted = 1 | target = 1
sum = 12 | predicted = 0 | target = 0
sum = 17 | predicted = 1 | target = 1
sum = 18 | predicted = 0 | target = 0
sum = 16 | predicted = 0 | target = 0
sum = 13 | predicted = 1 | target = 1
sum = 17 | predicted = 1 | target = 1
sum = 22 | predicted = 0 | target = 0
sum = 12 | predicted = 0 | target = 0


In [36]:
class EasyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, 16),
            nn.ReLU(),
            nn.Linear(16, 2)
        )

    def forward(self, x):
        return self.net(x)


easy_model = EasyModel().to(device)
optimizer = torch.optim.AdamW(easy_model.parameters(), lr=1e-3)

# Simple deterministic dataset:
# x < 0 -> class 0
# x >= 0 -> class 1
x_easy = torch.randn(5000, 1, device=device)
y_easy = (x_easy[:, 0] >= 0).long()

for step in range(501):
    idx = torch.randint(0, len(x_easy), (128,), device=device)

    batch_x = x_easy[idx]
    batch_y = y_easy[idx]

    logits = easy_model(batch_x)
    loss = F.cross_entropy(logits, batch_y)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % 100 == 0:
        accuracy = (
            logits.argmax(dim=-1) == batch_y
        ).float().mean().item() * 100

        print(
            f"Step {step:4d} | "
            f"Loss {loss.item():.4f} | "
            f"Accuracy {accuracy:.2f}%"
        )

Step    0 | Loss 0.7443 | Accuracy 40.62%
Step  100 | Loss 0.4695 | Accuracy 85.94%
Step  200 | Loss 0.2950 | Accuracy 99.22%
Step  300 | Loss 0.2027 | Accuracy 100.00%
Step  400 | Loss 0.1331 | Accuracy 100.00%
Step  500 | Loss 0.0919 | Accuracy 99.22%


In [37]:
# Cell: RLT recurrent state container

from dataclasses import dataclass
from typing import Optional
import torch


@dataclass
class RLTState:
    """
    Recurrent state carried through the sequence.

    s      : recurrent latent state
    memory : encoder KV memory
    cache  : decoder sliding-window KV cache
    pos    : current sequence position
    """

    s: torch.Tensor
    memory: Optional[object] = None
    cache: Optional[object] = None
    pos: int = 0


# Test
B = 4
D = 32

s0 = torch.zeros(B, D, device=device)

state = RLTState(
    s=s0,
    memory=None,
    cache=None,
    pos=0
)

print("State:", state)
print("s shape:", state.s.shape)
print("Initial position:", state.pos)

State: RLTState(s=tensor([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0.]], device='cuda:0'), memory=None, cache=None, pos=0)
s shape: torch.Size([4, 32])
Initial position: 0


In [38]:
# Cell: RLT state initialization

def init_rlt_state(batch_size, d_model, device):
    """
    Initialize the recurrent state H0.

    H0 = (s*, empty)
    For now, s* is represented by a zero vector.
    """

    s_star = torch.zeros(
        batch_size,
        d_model,
        device=device
    )

    return RLTState(
        s=s_star,
        memory=None,
        cache=None,
        pos=0
    )


# Test
state = init_rlt_state(
    batch_size=4,
    d_model=32,
    device=device
)

print("s* shape:", state.s.shape)
print("memory:", state.memory)
print("cache:", state.cache)
print("position:", state.pos)

s* shape: torch.Size([4, 32])
memory: None
cache: None
position: 0


In [39]:
# Cell: RLT causal encoder

class RLTEncoder(nn.Module):
    def __init__(
        self,
        d_model=32,
        n_heads=4,
        ff_dim=128,
        n_layers=1
    ):
        super().__init__()

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=ff_dim,
            dropout=0.0,
            activation="gelu",
            batch_first=True,
            norm_first=False
        )

        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=n_layers
        )

        self.norm = RMSNorm(d_model)

    def forward(self, x):
        """
        x: [B, T, D]

        Returns:
            encoder representations e_1:T
        """

        B, T, D = x.shape

        # Causal mask:
        # position t can only see positions <= t
        causal_mask = torch.triu(
            torch.ones(
                T, T,
                device=x.device,
                dtype=torch.bool
            ),
            diagonal=1
        )

        e = self.encoder(
            x,
            mask=causal_mask
        )

        e = self.norm(e)

        return e


# Test
B = 4
T = 32
D = 32

encoder = RLTEncoder(
    d_model=D,
    n_heads=4,
    ff_dim=128,
    n_layers=1
).to(device)

x = torch.randn(B, T, D, device=device)

e = encoder(x)

print("Input shape :", x.shape)
print("Output shape:", e.shape)
print("Finite      :", torch.isfinite(e).all().item())

Input shape : torch.Size([4, 32, 32])
Output shape: torch.Size([4, 32, 32])
Finite      : True


In [40]:
# Cell: Store encoder memory in RLT state

def encode_and_store(encoder, x, state):
    """
    Causal encoder prefill.

    x:
        [B, T, D]

    state:
        initial RLTState

    Returns:
        updated RLTState containing encoder memory
    """

    memory = encoder(x)

    state.memory = memory
    state.pos = 0

    return state


# Test
state = init_rlt_state(
    batch_size=B,
    d_model=D,
    device=device
)

state = encode_and_store(
    encoder,
    x,
    state
)

print("Encoder memory shape:", state.memory.shape)
print("Recurrent state shape:", state.s.shape)
print("Position:", state.pos)

Encoder memory shape: torch.Size([4, 32, 32])
Recurrent state shape: torch.Size([4, 32])
Position: 0


In [41]:
# Cell: Current encoder representation

def get_encoder_state(memory, t):
    """
    Get the encoder representation e_t.

    memory: [B, T, D]
    t: zero-based position
    """

    return memory[:, t, :]


# Test
t = 0

e_t = get_encoder_state(
    state.memory,
    t
)

print("Memory shape :", state.memory.shape)
print("e_t shape   :", e_t.shape)
print("Position     :", t)
print("Finite       :", torch.isfinite(e_t).all().item())

Memory shape : torch.Size([4, 32, 32])
e_t shape   : torch.Size([4, 32])
Position     : 0
Finite       : True


In [42]:
# Cell: RLT state-token merge

class RLTMerge(nn.Module):
    def __init__(self, d_model):
        super().__init__()

        self.proj = nn.Linear(
            2 * d_model,
            d_model
        )

        self.norm = RMSNorm(d_model)

    def forward(self, e_t, s_prev):
        """
        e_t   : [B, D] current encoder representation
        s_prev: [B, D] previous recurrent state

        returns:
            merged representation [B, D]
        """

        merged = torch.cat(
            [e_t, s_prev],
            dim=-1
        )

        merged = self.proj(merged)
        merged = self.norm(merged)

        return merged


# Test
merge = RLTMerge(D).to(device)

s_prev = state.s
merged = merge(e_t, s_prev)

print("e_t shape    :", e_t.shape)
print("s_prev shape :", s_prev.shape)
print("merged shape :", merged.shape)
print("Finite       :", torch.isfinite(merged).all().item())

e_t shape    : torch.Size([4, 32])
s_prev shape : torch.Size([4, 32])
merged shape : torch.Size([4, 32])
Finite       : True


In [44]:
# Cell: Test decoder SWA cache through recurrent steps

D = 32
WINDOW = 8

swa = SlidingWindowAttentionV2(
    d_model=D,
    window_size=WINDOW
).to(device)

B = 4
cache = None

for t in range(12):

    q = torch.randn(B, 1, D, device=device)

    output, cache = swa(q, cache)

    # Cache is a tuple in our implementation
    if isinstance(cache, tuple):
        cache_shapes = [
            tuple(c.shape) for c in cache
            if torch.is_tensor(c)
        ]
    else:
        cache_shapes = tuple(cache.shape)

    print(
        f"Step {t+1:2d} | "
        f"Output {tuple(output.shape)} | "
        f"Cache {cache_shapes}"
    )

Step  1 | Output (4, 1, 32) | Cache [(4, 1, 32), (4, 1, 32)]
Step  2 | Output (4, 1, 32) | Cache [(4, 2, 32), (4, 2, 32)]
Step  3 | Output (4, 1, 32) | Cache [(4, 3, 32), (4, 3, 32)]
Step  4 | Output (4, 1, 32) | Cache [(4, 4, 32), (4, 4, 32)]
Step  5 | Output (4, 1, 32) | Cache [(4, 5, 32), (4, 5, 32)]
Step  6 | Output (4, 1, 32) | Cache [(4, 6, 32), (4, 6, 32)]
Step  7 | Output (4, 1, 32) | Cache [(4, 7, 32), (4, 7, 32)]
Step  8 | Output (4, 1, 32) | Cache [(4, 8, 32), (4, 8, 32)]
Step  9 | Output (4, 1, 32) | Cache [(4, 8, 32), (4, 8, 32)]
Step 10 | Output (4, 1, 32) | Cache [(4, 8, 32), (4, 8, 32)]
Step 11 | Output (4, 1, 32) | Cache [(4, 8, 32), (4, 8, 32)]
Step 12 | Output (4, 1, 32) | Cache [(4, 8, 32), (4, 8, 32)]


In [45]:
# Cell: One recurrent decoder transition

class RLTTransition(nn.Module):
    def __init__(self, d_model=32, window_size=8):
        super().__init__()

        self.merge = RLTMerge(d_model)

        self.swa = SlidingWindowAttentionV2(
            d_model=d_model,
            window_size=window_size
        )

        self.state_update = nn.GRUCell(
            d_model,
            d_model
        )

    def forward(self, e_t, s_prev, cache):
        """
        One recurrent transition.

        e_t    : [B, D]
        s_prev : [B, D]
        cache  : (K_cache, V_cache) or None

        Returns:
            s_t
            new_cache
        """

        # 1. Merge current encoder representation
        #    with previous recurrent state
        merged = self.merge(e_t, s_prev)

        # 2. Apply sliding-window attention
        attn_input = merged.unsqueeze(1)

        attn_output, new_cache = self.swa(
            attn_input,
            cache
        )

        attn_output = attn_output.squeeze(1)

        # 3. Recurrent state update
        s_t = self.state_update(
            attn_output,
            s_prev
        )

        return s_t, new_cache


# Test one transition
transition = RLTTransition(
    d_model=D,
    window_size=8
).to(device)

s_prev = state.s
cache = None

e_t = state.memory[:, 0, :]

s_t, cache = transition(
    e_t,
    s_prev,
    cache
)

print("Previous state:", s_prev.shape)
print("Current e_t    :", e_t.shape)
print("New state      :", s_t.shape)

print(
    "Cache shapes   :",
    [tuple(c.shape) for c in cache]
)

print(
    "Finite state   :",
    torch.isfinite(s_t).all().item()
)

Previous state: torch.Size([4, 32])
Current e_t    : torch.Size([4, 32])
New state      : torch.Size([4, 32])
Cache shapes   : [(4, 1, 32), (4, 1, 32)]
Finite state   : True


In [46]:
# Cell: Unroll recurrent decoder across a sequence

B = 4
T = 12
D = 32

# Fresh encoder input
x = torch.randn(B, T, D, device=device)

# Causal encoder
memory = encoder(x)

# Initial recurrent state
state = init_rlt_state(
    batch_size=B,
    d_model=D,
    device=device
)

# Transition module
transition = RLTTransition(
    d_model=D,
    window_size=8
).to(device)

cache = None

states = []

for t in range(T):

    # Current encoder representation e_t
    e_t = memory[:, t, :]

    # Recurrent transition
    s_t, cache = transition(
        e_t,
        state.s,
        cache
    )

    # Store state
    states.append(s_t)

    # Carry state forward
    state.s = s_t
    state.pos = t + 1

    print(
        f"Step {t+1:2d} | "
        f"state {tuple(s_t.shape)} | "
        f"cache K,V = "
        f"{tuple(cache[0].shape)}, "
        f"{tuple(cache[1].shape)}"
    )

print("\nFinal state:", state.s.shape)
print("Final position:", state.pos)

Step  1 | state (4, 32) | cache K,V = (4, 1, 32), (4, 1, 32)
Step  2 | state (4, 32) | cache K,V = (4, 2, 32), (4, 2, 32)
Step  3 | state (4, 32) | cache K,V = (4, 3, 32), (4, 3, 32)
Step  4 | state (4, 32) | cache K,V = (4, 4, 32), (4, 4, 32)
Step  5 | state (4, 32) | cache K,V = (4, 5, 32), (4, 5, 32)
Step  6 | state (4, 32) | cache K,V = (4, 6, 32), (4, 6, 32)
Step  7 | state (4, 32) | cache K,V = (4, 7, 32), (4, 7, 32)
Step  8 | state (4, 32) | cache K,V = (4, 8, 32), (4, 8, 32)
Step  9 | state (4, 32) | cache K,V = (4, 8, 32), (4, 8, 32)
Step 10 | state (4, 32) | cache K,V = (4, 8, 32), (4, 8, 32)
Step 11 | state (4, 32) | cache K,V = (4, 8, 32), (4, 8, 32)
Step 12 | state (4, 32) | cache K,V = (4, 8, 32), (4, 8, 32)

Final state: torch.Size([4, 32])
Final position: 12


In [47]:
# Cell: Verify full BPTT through recurrent state + SWA cache

transition.zero_grad()
encoder.zero_grad()

B = 2
T = 12
D = 32

x = torch.randn(B, T, D, device=device)

# Encoder
memory = encoder(x)

# Initial state
state = init_rlt_state(
    batch_size=B,
    d_model=D,
    device=device
)

cache = None
states = []

for t in range(T):

    e_t = memory[:, t, :]

    s_t, cache = transition(
        e_t,
        state.s,
        cache
    )

    states.append(s_t)
    state.s = s_t

# Use final recurrent state as the loss source
loss = state.s.pow(2).mean()

loss.backward()

print("Loss:", loss.item())

# Check gradients
total = 0
with_grad = 0
nonzero = 0

for name, param in transition.named_parameters():
    if param.requires_grad:
        total += 1

        if param.grad is not None:
            with_grad += 1

            if param.grad.abs().sum().item() > 0:
                nonzero += 1

print("Transition parameters:", total)
print("With gradients:", with_grad)
print("Non-zero gradients:", nonzero)

# Encoder gradient check
encoder_total = 0
encoder_with_grad = 0
encoder_nonzero = 0

for name, param in encoder.named_parameters():
    if param.requires_grad:
        encoder_total += 1

        if param.grad is not None:
            encoder_with_grad += 1

            if param.grad.abs().sum().item() > 0:
                encoder_nonzero += 1

print("\nEncoder parameters:", encoder_total)
print("With gradients:", encoder_with_grad)
print("Non-zero gradients:", encoder_nonzero)

Loss: 0.12532144784927368
Transition parameters: 12
With gradients: 12
Non-zero gradients: 12

Encoder parameters: 13
With gradients: 13
Non-zero gradients: 13


In [48]:
# Cell: Verify gradient flow across recurrent steps

transition.zero_grad()
encoder.zero_grad()

B = 2
T = 12
D = 32

x = torch.randn(B, T, D, device=device)

memory = encoder(x)

state = init_rlt_state(
    batch_size=B,
    d_model=D,
    device=device
)

cache = None
states = []

for t in range(T):
    e_t = memory[:, t, :]

    s_t, cache = transition(
        e_t,
        state.s,
        cache
    )

    # Keep gradient on every recurrent state
    s_t.retain_grad()

    states.append(s_t)
    state.s = s_t

loss = state.s.pow(2).mean()
loss.backward()

print("Loss:", loss.item())

print("\nGradient magnitude of each recurrent state:")

for t, s in enumerate(states, start=1):
    grad_mag = s.grad.abs().mean().item()
    print(f"Step {t:2d}: {grad_mag:.8f}")

Loss: 0.11575715243816376

Gradient magnitude of each recurrent state:
Step  1: 0.00028243
Step  2: 0.00041450
Step  3: 0.00057442
Step  4: 0.00070537
Step  5: 0.00079420
Step  6: 0.00099595
Step  7: 0.00132659
Step  8: 0.00162734
Step  9: 0.00218457
Step 10: 0.00268297
Step 11: 0.00425633
Step 12: 0.00910724


In [49]:
# Cell: Complete RLT Model Wrapper

class RLTModel(nn.Module):
    def __init__(
        self,
        vocab_size,
        d_model=32,
        n_heads=4,
        ff_dim=128,
        window_size=8,
        max_seq_len=128,
        num_classes=None,
    ):
        super().__init__()

        if num_classes is None:
            num_classes = vocab_size

        self.d_model = d_model
        self.window_size = window_size
        self.max_seq_len = max_seq_len

        # Token embedding
        self.token_embedding = nn.Embedding(
            vocab_size,
            d_model
        )

        # Learned positional embedding
        self.position_embedding = nn.Embedding(
            max_seq_len,
            d_model
        )

        # Causal encoder
        self.encoder = RLTEncoder(
            d_model=d_model,
            n_heads=n_heads,
            ff_dim=ff_dim
        )

        # Shared recurrent decoder transition
        self.transition = RLTTransition(
            d_model=d_model,
            window_size=window_size
        )

        # Final normalization
        self.output_norm = RMSNorm(d_model)

        # Output head
        self.output_head = nn.Linear(
            d_model,
            num_classes
        )

    def forward(self, input_ids):
        """
        input_ids:
            [B, T]

        returns:
            logits: [B, num_classes]
            final_state: [B, D]
        """

        B, T = input_ids.shape

        assert T <= self.max_seq_len, (
            f"Sequence length {T} exceeds "
            f"max_seq_len={self.max_seq_len}"
        )

        # ------------------------------------------------
        # 1. Token + position embeddings
        # ------------------------------------------------

        positions = torch.arange(
            T,
            device=input_ids.device
        )

        x = self.token_embedding(input_ids)

        x = x + self.position_embedding(
            positions
        ).unsqueeze(0)

        # ------------------------------------------------
        # 2. Causal encoder
        # ------------------------------------------------

        memory = self.encoder(x)

        # ------------------------------------------------
        # 3. Initialize recurrent state
        # ------------------------------------------------

        state = init_rlt_state(
            batch_size=B,
            d_model=self.d_model,
            device=input_ids.device
        )

        cache = None

        # ------------------------------------------------
        # 4. Recurrent decoder
        # ------------------------------------------------

        for t in range(T):

            e_t = memory[:, t, :]

            s_t, cache = self.transition(
                e_t,
                state.s,
                cache
            )

            state.s = s_t
            state.pos = t + 1

        # ------------------------------------------------
        # 5. Final output
        # ------------------------------------------------

        final_state = self.output_norm(
            state.s
        )

        logits = self.output_head(
            final_state
        )

        return logits, final_state

In [50]:
# Cell: Instantiate complete RLT

model = RLTModel(
    vocab_size=2,
    d_model=32,
    n_heads=4,
    ff_dim=128,
    window_size=8,
    max_seq_len=128,
    num_classes=2
).to(device)

print(model)

RLTModel(
  (token_embedding): Embedding(2, 32)
  (position_embedding): Embedding(128, 32)
  (encoder): RLTEncoder(
    (encoder): TransformerEncoder(
      (layers): ModuleList(
        (0): TransformerEncoderLayer(
          (self_attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=32, out_features=32, bias=True)
          )
          (linear1): Linear(in_features=32, out_features=128, bias=True)
          (dropout): Dropout(p=0.0, inplace=False)
          (linear2): Linear(in_features=128, out_features=32, bias=True)
          (norm1): LayerNorm((32,), eps=1e-05, elementwise_affine=True)
          (norm2): LayerNorm((32,), eps=1e-05, elementwise_affine=True)
          (dropout1): Dropout(p=0.0, inplace=False)
          (dropout2): Dropout(p=0.0, inplace=False)
        )
      )
    )
    (norm): RMSNorm()
  )
  (transition): RLTTransition(
    (merge): RLTMerge(
      (proj): Linear(in_features=64, out_features=32, bias=True)
      (norm): 

In [51]:
# Cell: Parameter count

total_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print(f"Trainable parameters: {total_params:,}")

Trainable parameters: 29,570


In [52]:
# Cell: Forward-pass test

B = 4
T = 32

input_ids = torch.randint(
    0,
    2,
    (B, T),
    device=device
)

logits, final_state = model(input_ids)

print("Input:", input_ids.shape)
print("Logits:", logits.shape)
print("Final state:", final_state.shape)
print("Finite logits:", torch.isfinite(logits).all().item())
print("Finite state:", torch.isfinite(final_state).all().item())

Input: torch.Size([4, 32])
Logits: torch.Size([4, 2])
Final state: torch.Size([4, 32])
Finite logits: True
Finite state: True


In [53]:
# Cell: Parameter breakdown

print("Parameter breakdown:\n")

for name, module in model.named_children():
    params = sum(p.numel() for p in module.parameters())
    print(f"{name:20s}: {params:8,d}")

print("-" * 32)

total = sum(p.numel() for p in model.parameters())
print(f"{'TOTAL':20s}: {total:8,d}")

Parameter breakdown:

token_embedding     :       64
position_embedding  :    4,096
encoder             :   12,736
transition          :   12,576
output_norm         :       32
output_head         :       66
--------------------------------
TOTAL               :   29,570


In [54]:
# Cell: Paper-faithful Gated Merge
# Equations (2.9) - (2.11)

class RLTGatedMerge(nn.Module):
    def __init__(self, d_model, alpha=0.1):
        super().__init__()

        self.d_model = d_model
        self.alpha = alpha

        # Equation (2.9)
        self.state_norm = RMSNorm(d_model)

        # Equation (2.10)
        # Wg: d x 2d
        self.gate = nn.Linear(
            2 * d_model,
            d_model
        )

        # Equation (2.11)
        # Ws: d x d
        self.state_proj = nn.Linear(
            d_model,
            d_model,
            bias=False
        )

    def forward(self, e_t, s_prev):

        # (2.9)
        r_prev = self.state_norm(s_prev)

        # (2.10)
        merged_input = torch.cat(
            [e_t, r_prev],
            dim=-1
        )

        g_t = torch.sigmoid(
            self.gate(merged_input)
        )

        # (2.11)
        feedback = self.state_proj(r_prev)

        u_t = e_t + self.alpha * g_t * feedback

        return u_t

In [55]:
# Cell: Test Gated Merge

merge = RLTGatedMerge(
    d_model=32,
    alpha=0.1
).to(device)

B = 4
D = 32

e_t = torch.randn(B, D, device=device)
s_prev = torch.randn(B, D, device=device)

u_t = merge(e_t, s_prev)

print("e_t:", e_t.shape)
print("s_prev:", s_prev.shape)
print("u_t:", u_t.shape)

print(
    "Finite output:",
    torch.isfinite(u_t).all().item()
)

print(
    "Parameters:",
    sum(p.numel() for p in merge.parameters())
)

e_t: torch.Size([4, 32])
s_prev: torch.Size([4, 32])
u_t: torch.Size([4, 32])
Finite output: True
Parameters: 3136


In [56]:
class EncoderCrossAttention(nn.Module):
    def __init__(self, d_model=32, n_heads=4):
        super().__init__()

        self.attn = nn.MultiheadAttention(
            embed_dim=d_model,
            num_heads=n_heads,
            batch_first=True,
            dropout=0.0
        )

        self.norm = RMSNorm(d_model)

    def forward(self, query, memory):
        """
        query:
            [B, 1, D]

        memory:
            [B, T, D]

        Returns:
            [B, 1, D]
        """

        q = self.norm(query)

        output, _ = self.attn(
            q,
            memory,
            memory,
            need_weights=False
        )

        return query + output

In [57]:
class RLTFFN(nn.Module):
    def __init__(self, d_model=32, ff_dim=128):
        super().__init__()

        self.norm = RMSNorm(d_model)

        self.fc1 = nn.Linear(
            d_model,
            ff_dim
        )

        self.fc2 = nn.Linear(
            ff_dim,
            d_model
        )

    def forward(self, x):
        residual = x

        x = self.norm(x)
        x = self.fc1(x)
        x = F.gelu(x)
        x = self.fc2(x)

        return residual + x

In [58]:
class RLTDecoderTransitionV2(nn.Module):
    def __init__(
        self,
        d_model=32,
        n_heads=4,
        ff_dim=128,
        window_size=8
    ):
        super().__init__()

        self.merge = RLTGatedMerge(
            d_model=d_model,
            alpha=0.1
        )

        self.swa = SlidingWindowAttentionV2(
            d_model=d_model,
            window_size=window_size
        )

        self.cross_attn = EncoderCrossAttention(
            d_model=d_model,
            n_heads=n_heads
        )

        self.ffn = RLTFFN(
            d_model=d_model,
            ff_dim=ff_dim
        )

    def forward(
        self,
        e_t,
        s_prev,
        memory,
        cache
    ):

        # ------------------------------------------
        # 1. Gated state-token merge
        # ------------------------------------------

        u_t = self.merge(
            e_t,
            s_prev
        )

        # [B,D] -> [B,1,D]
        x = u_t.unsqueeze(1)

        # ------------------------------------------
        # 2. Sliding-window decoder attention
        # ------------------------------------------

        x, new_cache = self.swa(
            x,
            cache
        )

        # ------------------------------------------
        # 3. Cross-attention to encoder memory
        # ------------------------------------------

        x = self.cross_attn(
            x,
            memory
        )

        # ------------------------------------------
        # 4. Feed-forward network
        # ------------------------------------------

        x = self.ffn(x)

        # [B,1,D] -> [B,D]
        s_t = x.squeeze(1)

        return s_t, new_cache

In [59]:
transition_v2 = RLTDecoderTransitionV2(
    d_model=32,
    n_heads=4,
    ff_dim=128,
    window_size=8
).to(device)

B = 4
T = 32
D = 32

memory = torch.randn(
    B, T, D,
    device=device
)

e_t = memory[:, 0, :]

s_prev = torch.zeros(
    B, D,
    device=device
)

cache = None

s_t, new_cache = transition_v2(
    e_t,
    s_prev,
    memory,
    cache
)

print("s_prev:", s_prev.shape)
print("e_t:", e_t.shape)
print("memory:", memory.shape)
print("s_t:", s_t.shape)

print(
    "K cache:",
    new_cache[0].shape
)

print(
    "V cache:",
    new_cache[1].shape
)

print(
    "Finite state:",
    torch.isfinite(s_t).all().item()
)

s_prev: torch.Size([4, 32])
e_t: torch.Size([4, 32])
memory: torch.Size([4, 32, 32])
s_t: torch.Size([4, 32])
K cache: torch.Size([4, 1, 32])
V cache: torch.Size([4, 1, 32])
Finite state: True


In [60]:
# Cell: BPTT test for RLTDecoderTransitionV2

transition_v2.zero_grad()

B = 2
T = 12
D = 32

memory = torch.randn(
    B, T, D,
    device=device,
    requires_grad=True
)

s_prev = torch.zeros(
    B, D,
    device=device
)

cache = None
states = []

for t in range(T):

    e_t = memory[:, t, :]

    s_t, cache = transition_v2(
        e_t,
        s_prev,
        memory,
        cache
    )

    s_t.retain_grad()

    states.append(s_t)

    s_prev = s_t


loss = states[-1].pow(2).mean()

loss.backward()

print("Loss:", loss.item())

print("\nState gradients:")

for t, s in enumerate(states, start=1):
    print(
        f"Step {t:2d}: "
        f"{s.grad.abs().mean().item():.8f}"
    )


print("\nParameter gradients:")

total = 0
with_grad = 0
nonzero = 0

for name, param in transition_v2.named_parameters():

    if param.requires_grad:

        total += 1

        if param.grad is not None:
            with_grad += 1

            if param.grad.abs().sum().item() > 0:
                nonzero += 1

print("Total:", total)
print("With gradients:", with_grad)
print("Non-zero:", nonzero)

Loss: 1.110795021057129

State gradients:
Step  1: 0.00000005
Step  2: 0.00000006
Step  3: 0.00000051
Step  4: 0.00000767
Step  5: 0.00000764
Step  6: 0.00000832
Step  7: 0.00000660
Step  8: 0.00000760
Step  9: 0.00001114
Step 10: 0.00000867
Step 11: 0.00029677
Step 12: 0.02686781

Parameter gradients:
Total: 19
With gradients: 19
Non-zero: 19


In [61]:
def get_causal_memory_prefix(memory, t):
    """
    Return encoder memory visible at recurrent step t.

    t = 0  -> positions 0
    t = 1  -> positions 0,1
    t = 2  -> positions 0,1,2
    ...
    """
    return memory[:, :t + 1, :]

In [62]:
# Cell: Test causal encoder-memory prefixes

B = 2
T = 8
D = 32

memory = torch.randn(
    B, T, D,
    device=device
)

for t in range(T):
    prefix = get_causal_memory_prefix(memory, t)

    print(
        f"Step {t+1:2d} -> "
        f"prefix shape: {prefix.shape}"
    )

Step  1 -> prefix shape: torch.Size([2, 1, 32])
Step  2 -> prefix shape: torch.Size([2, 2, 32])
Step  3 -> prefix shape: torch.Size([2, 3, 32])
Step  4 -> prefix shape: torch.Size([2, 4, 32])
Step  5 -> prefix shape: torch.Size([2, 5, 32])
Step  6 -> prefix shape: torch.Size([2, 6, 32])
Step  7 -> prefix shape: torch.Size([2, 7, 32])
Step  8 -> prefix shape: torch.Size([2, 8, 32])


In [63]:
# Cell: V2 transition with causal encoder prefixes

B = 2
T = 8
D = 32

memory = torch.randn(
    B, T, D,
    device=device
)

s_prev = torch.zeros(
    B, D,
    device=device
)

cache = None

for t in range(T):

    e_t = memory[:, t, :]

    # Only M_{<=t}
    memory_prefix = memory[:, :t + 1, :]

    s_t, cache = transition_v2(
        e_t,
        s_prev,
        memory_prefix,
        cache
    )

    s_prev = s_t

    print(
        f"Step {t+1:2d} | "
        f"Memory {memory_prefix.shape} | "
        f"State {s_t.shape} | "
        f"Cache {cache[0].shape}"
    )

Step  1 | Memory torch.Size([2, 1, 32]) | State torch.Size([2, 32]) | Cache torch.Size([2, 1, 32])
Step  2 | Memory torch.Size([2, 2, 32]) | State torch.Size([2, 32]) | Cache torch.Size([2, 2, 32])
Step  3 | Memory torch.Size([2, 3, 32]) | State torch.Size([2, 32]) | Cache torch.Size([2, 3, 32])
Step  4 | Memory torch.Size([2, 4, 32]) | State torch.Size([2, 32]) | Cache torch.Size([2, 4, 32])
Step  5 | Memory torch.Size([2, 5, 32]) | State torch.Size([2, 32]) | Cache torch.Size([2, 5, 32])
Step  6 | Memory torch.Size([2, 6, 32]) | State torch.Size([2, 32]) | Cache torch.Size([2, 6, 32])
Step  7 | Memory torch.Size([2, 7, 32]) | State torch.Size([2, 32]) | Cache torch.Size([2, 7, 32])
Step  8 | Memory torch.Size([2, 8, 32]) | State torch.Size([2, 32]) | Cache torch.Size([2, 8, 32])


In [64]:
# Cell: Verify SWA cache eviction

B = 2
T = 12
D = 32

memory = torch.randn(
    B, T, D,
    device=device
)

s_prev = torch.zeros(
    B, D,
    device=device
)

cache = None

for t in range(T):

    e_t = memory[:, t, :]
    memory_prefix = memory[:, :t + 1, :]

    s_t, cache = transition_v2(
        e_t,
        s_prev,
        memory_prefix,
        cache
    )

    s_prev = s_t

    print(
        f"Step {t+1:2d} | "
        f"Cache length: {cache[0].shape[1]}"
    )

Step  1 | Cache length: 1
Step  2 | Cache length: 2
Step  3 | Cache length: 3
Step  4 | Cache length: 4
Step  5 | Cache length: 5
Step  6 | Cache length: 6
Step  7 | Cache length: 7
Step  8 | Cache length: 8
Step  9 | Cache length: 8
Step 10 | Cache length: 8
Step 11 | Cache length: 8
Step 12 | Cache length: 8


In [65]:
# Cell: Complete RLT Model V2

class RLTModelV2(nn.Module):
    def __init__(
        self,
        vocab_size,
        d_model=32,
        n_heads=4,
        ff_dim=128,
        window_size=8,
        max_seq_len=128,
        num_classes=None,
    ):
        super().__init__()

        if num_classes is None:
            num_classes = vocab_size

        self.d_model = d_model
        self.max_seq_len = max_seq_len

        # Token + position embeddings
        self.token_embedding = nn.Embedding(
            vocab_size,
            d_model
        )

        self.position_embedding = nn.Embedding(
            max_seq_len,
            d_model
        )

        # Causal encoder
        self.encoder = RLTEncoder(
            d_model=d_model,
            n_heads=n_heads,
            ff_dim=ff_dim
        )

        # NEW paper-structure transition
        self.transition = RLTDecoderTransitionV2(
            d_model=d_model,
            n_heads=n_heads,
            ff_dim=ff_dim,
            window_size=window_size
        )

        # Output
        self.output_norm = RMSNorm(d_model)

        self.output_head = nn.Linear(
            d_model,
            num_classes
        )

    def forward(self, input_ids):

        B, T = input_ids.shape

        assert T <= self.max_seq_len

        # ----------------------------------------
        # 1. Token + positional embeddings
        # ----------------------------------------

        positions = torch.arange(
            T,
            device=input_ids.device
        )

        x = self.token_embedding(input_ids)

        x = x + self.position_embedding(
            positions
        ).unsqueeze(0)

        # ----------------------------------------
        # 2. Causal encoder
        # ----------------------------------------

        memory = self.encoder(x)

        # ----------------------------------------
        # 3. Initial recurrent state
        # ----------------------------------------

        s_prev = torch.zeros(
            B,
            self.d_model,
            device=input_ids.device,
            dtype=x.dtype
        )

        cache = None

        # ----------------------------------------
        # 4. Recurrent decoder
        # ----------------------------------------

        for t in range(T):

            # Current encoder representation
            e_t = memory[:, t, :]

            # IMPORTANT:
            # Decoder can only see M_{<=t}
            memory_prefix = memory[:, :t + 1, :]

            # Shared transition
            s_t, cache = self.transition(
                e_t,
                s_prev,
                memory_prefix,
                cache
            )

            s_prev = s_t

        # ----------------------------------------
        # 5. Final prediction
        # ----------------------------------------

        final_state = self.output_norm(
            s_prev
        )

        logits = self.output_head(
            final_state
        )

        return logits, final_state

In [66]:
# Cell: Instantiate RLT V2

model_v2 = RLTModelV2(
    vocab_size=2,
    d_model=32,
    n_heads=4,
    ff_dim=128,
    window_size=8,
    max_seq_len=128,
    num_classes=2
).to(device)

print(model_v2)

RLTModelV2(
  (token_embedding): Embedding(2, 32)
  (position_embedding): Embedding(128, 32)
  (encoder): RLTEncoder(
    (encoder): TransformerEncoder(
      (layers): ModuleList(
        (0): TransformerEncoderLayer(
          (self_attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=32, out_features=32, bias=True)
          )
          (linear1): Linear(in_features=32, out_features=128, bias=True)
          (dropout): Dropout(p=0.0, inplace=False)
          (linear2): Linear(in_features=128, out_features=32, bias=True)
          (norm1): LayerNorm((32,), eps=1e-05, elementwise_affine=True)
          (norm2): LayerNorm((32,), eps=1e-05, elementwise_affine=True)
          (dropout1): Dropout(p=0.0, inplace=False)
          (dropout2): Dropout(p=0.0, inplace=False)
        )
      )
    )
    (norm): RMSNorm()
  )
  (transition): RLTDecoderTransitionV2(
    (merge): RLTGatedMerge(
      (state_norm): RMSNorm()
      (gate): Linear(in_features=

In [67]:
# Cell: Parameter count

total_params = sum(
    p.numel()
    for p in model_v2.parameters()
    if p.requires_grad
)

print(f"Trainable parameters: {total_params:,}")

Trainable parameters: 36,898


In [68]:
# Cell: Complete V2 forward test

B = 4
T = 32

input_ids = torch.randint(
    0,
    2,
    (B, T),
    device=device
)

logits, final_state = model_v2(input_ids)

print("Input:", input_ids.shape)
print("Logits:", logits.shape)
print("Final state:", final_state.shape)

print(
    "Finite logits:",
    torch.isfinite(logits).all().item()
)

print(
    "Finite state:",
    torch.isfinite(final_state).all().item()
)

Input: torch.Size([4, 32])
Logits: torch.Size([4, 2])
Final state: torch.Size([4, 32])
Finite logits: True
Finite state: True


In [69]:
# Cell: Complete-model BPTT test

model_v2.train()

B = 2
T = 12

input_ids = torch.randint(
    0,
    2,
    (B, T),
    device=device
)

targets = torch.randint(
    0,
    2,
    (B,),
    device=device
)

# Forward
logits, final_state = model_v2(input_ids)

# Loss
loss = F.cross_entropy(
    logits,
    targets
)

# Backward
model_v2.zero_grad(set_to_none=True)
loss.backward()

print("Loss:", loss.item())

# ----------------------------------------
# Check parameter gradients
# ----------------------------------------

total = 0
with_grad = 0
nonzero = 0

for name, param in model_v2.named_parameters():

    if param.requires_grad:

        total += 1

        if param.grad is not None:
            with_grad += 1

            if param.grad.abs().sum().item() > 0:
                nonzero += 1

print("Parameters:", total)
print("With gradients:", with_grad)
print("Non-zero gradients:", nonzero)

Loss: 1.086005687713623
Parameters: 37
With gradients: 37
Non-zero gradients: 37


In [70]:
# Cell: Gradient magnitude inspection

print("Gradient magnitudes")
print("-" * 60)

for name, param in model_v2.named_parameters():

    if param.grad is not None:
        grad_norm = param.grad.norm().item()

        print(
            f"{name:45s} "
            f"{grad_norm:.8e}"
        )

Gradient magnitudes
------------------------------------------------------------
token_embedding.weight                        3.30087811e-01
position_embedding.weight                     2.89064825e-01
encoder.encoder.layers.0.self_attn.in_proj_weight 1.20935071e+00
encoder.encoder.layers.0.self_attn.in_proj_bias 2.06734553e-01
encoder.encoder.layers.0.self_attn.out_proj.weight 1.18934643e+00
encoder.encoder.layers.0.self_attn.out_proj.bias 3.25394303e-01
encoder.encoder.layers.0.linear1.weight       9.31177258e-01
encoder.encoder.layers.0.linear1.bias         1.75857306e-01
encoder.encoder.layers.0.linear2.weight       2.60225272e+00
encoder.encoder.layers.0.linear2.bias         5.52680552e-01
encoder.encoder.layers.0.norm1.weight         4.80504781e-01
encoder.encoder.layers.0.norm1.bias           5.72326720e-01
encoder.encoder.layers.0.norm2.weight         4.59339440e-01
encoder.encoder.layers.0.norm2.bias           5.38912654e-01
encoder.norm.weight                           4.628

In [71]:
# Cell: Training dataset

torch.manual_seed(42)

N = 10000
T = 32

X_train = torch.randint(
    0, 2,
    (N, T),
    device=device
)

# Target = parity of the sequence
y_train = X_train.sum(dim=1) % 2

print("X:", X_train.shape)
print("y:", y_train.shape)
print("Class 0:", (y_train == 0).sum().item())
print("Class 1:", (y_train == 1).sum().item())

X: torch.Size([10000, 32])
y: torch.Size([10000])
Class 0: 5064
Class 1: 4936


In [72]:
# Cell: Train / validation split

N_train = 8000

X_train_split = X_train[:N_train]
y_train_split = y_train[:N_train]

X_val = X_train[N_train:]
y_val = y_train[N_train:]

print("Training:", X_train_split.shape, y_train_split.shape)
print("Validation:", X_val.shape, y_val.shape)

Training: torch.Size([8000, 32]) torch.Size([8000])
Validation: torch.Size([2000, 32]) torch.Size([2000])


In [73]:
# Cell: Train RLTModelV2

import torch.nn.functional as F

# Fresh model for the actual experiment
model_v2 = RLTModelV2(
    vocab_size=2,
    d_model=32,
    n_heads=4,
    ff_dim=128,
    window_size=8,
    max_seq_len=128,
    num_classes=2
).to(device)

optimizer = torch.optim.AdamW(
    model_v2.parameters(),
    lr=3e-4,
    weight_decay=1e-2
)

batch_size = 64
steps = 1000

model_v2.train()

for step in range(steps):

    # Random mini-batch
    idx = torch.randint(
        0,
        X_train_split.size(0),
        (batch_size,),
        device=device
    )

    x_batch = X_train_split[idx]
    y_batch = y_train_split[idx]

    # Forward
    logits, _ = model_v2(x_batch)

    # Loss
    loss = F.cross_entropy(
        logits,
        y_batch
    )

    # Backprop
    optimizer.zero_grad(set_to_none=True)
    loss.backward()

    # Update
    optimizer.step()

    # Print progress
    if step % 100 == 0:

        with torch.no_grad():
            train_acc = (
                logits.argmax(dim=1) == y_batch
            ).float().mean().item() * 100

        print(
            f"Step {step:4d} | "
            f"Loss {loss.item():.4f} | "
            f"Batch Acc {train_acc:.2f}%"
        )

Step    0 | Loss 0.7161 | Batch Acc 54.69%
Step  100 | Loss 0.6992 | Batch Acc 39.06%
Step  200 | Loss 0.7035 | Batch Acc 45.31%
Step  300 | Loss 0.6944 | Batch Acc 56.25%
Step  400 | Loss 0.7071 | Batch Acc 45.31%
Step  500 | Loss 0.6951 | Batch Acc 50.00%
Step  600 | Loss 0.6909 | Batch Acc 51.56%
Step  700 | Loss 0.7050 | Batch Acc 45.31%
Step  800 | Loss 0.6934 | Batch Acc 50.00%
Step  900 | Loss 0.6891 | Batch Acc 57.81%


In [74]:
# Cell: Validation evaluation

model_v2.eval()

with torch.no_grad():
    val_logits, _ = model_v2(X_val)

    val_loss = F.cross_entropy(
        val_logits,
        y_val
    )

    val_pred = val_logits.argmax(dim=1)

    val_acc = (
        val_pred == y_val
    ).float().mean().item() * 100

print(f"Validation Loss: {val_loss.item():.4f}")
print(f"Validation Accuracy: {val_acc:.2f}%")

Validation Loss: 0.6928
Validation Accuracy: 52.20%


In [75]:
# Cell: Tiny-set overfitting test

model_tiny = RLTModelV2(
    vocab_size=2,
    d_model=32,
    n_heads=4,
    ff_dim=128,
    window_size=8,
    max_seq_len=128,
    num_classes=2
).to(device)

optimizer_tiny = torch.optim.AdamW(
    model_tiny.parameters(),
    lr=1e-3,
    weight_decay=0.0
)

X_tiny = X_train_split[:32]
y_tiny = y_train_split[:32]

model_tiny.train()

for step in range(1000):

    logits, _ = model_tiny(X_tiny)

    loss = F.cross_entropy(
        logits,
        y_tiny
    )

    optimizer_tiny.zero_grad(set_to_none=True)
    loss.backward()
    optimizer_tiny.step()

    if step % 100 == 0:
        acc = (
            logits.argmax(dim=1) == y_tiny
        ).float().mean().item() * 100

        print(
            f"Step {step:4d} | "
            f"Loss {loss.item():.4f} | "
            f"Accuracy {acc:.2f}%"
        )

Step    0 | Loss 0.7394 | Accuracy 50.00%
Step  100 | Loss 0.0022 | Accuracy 100.00%
Step  200 | Loss 0.0010 | Accuracy 100.00%
Step  300 | Loss 0.0006 | Accuracy 100.00%
Step  400 | Loss 0.0004 | Accuracy 100.00%
Step  500 | Loss 0.0003 | Accuracy 100.00%
Step  600 | Loss 0.0002 | Accuracy 100.00%
Step  700 | Loss 0.0002 | Accuracy 100.00%
Step  800 | Loss 0.0001 | Accuracy 100.00%
Step  900 | Loss 0.0001 | Accuracy 100.00%


In [76]:
# Cell: RLT Model with per-token recurrent states

class RLTModelV3(nn.Module):
    def __init__(
        self,
        vocab_size,
        d_model=32,
        n_heads=4,
        ff_dim=128,
        window_size=8,
        max_seq_len=128,
        num_classes=None,
    ):
        super().__init__()

        if num_classes is None:
            num_classes = vocab_size

        self.d_model = d_model
        self.max_seq_len = max_seq_len

        self.token_embedding = nn.Embedding(
            vocab_size,
            d_model
        )

        self.position_embedding = nn.Embedding(
            max_seq_len,
            d_model
        )

        self.encoder = RLTEncoder(
            d_model=d_model,
            n_heads=n_heads,
            ff_dim=ff_dim
        )

        self.transition = RLTDecoderTransitionV2(
            d_model=d_model,
            n_heads=n_heads,
            ff_dim=ff_dim,
            window_size=window_size
        )

        self.output_norm = RMSNorm(d_model)

        self.output_head = nn.Linear(
            d_model,
            num_classes
        )

    def forward(self, input_ids):

        B, T = input_ids.shape

        assert T <= self.max_seq_len

        # Token + position embeddings
        positions = torch.arange(
            T,
            device=input_ids.device
        )

        x = (
            self.token_embedding(input_ids)
            + self.position_embedding(positions).unsqueeze(0)
        )

        # Causal encoder
        memory = self.encoder(x)

        # Initial recurrent state
        s_prev = torch.zeros(
            B,
            self.d_model,
            device=input_ids.device,
            dtype=x.dtype
        )

        cache = None

        # Store every recurrent state
        states = []

        for t in range(T):

            e_t = memory[:, t, :]

            # Causal encoder prefix
            memory_prefix = memory[:, :t + 1, :]

            s_t, cache = self.transition(
                e_t,
                s_prev,
                memory_prefix,
                cache
            )

            states.append(s_t)
            s_prev = s_t

        # [B, T, D]
        states = torch.stack(states, dim=1)

        # Prediction from every recurrent state
        normalized_states = self.output_norm(states)

        logits = self.output_head(
            normalized_states
        )

        return logits, states

In [77]:
# Cell: V3 shape test

model_v3 = RLTModelV3(
    vocab_size=2,
    d_model=32,
    n_heads=4,
    ff_dim=128,
    window_size=8,
    max_seq_len=128,
    num_classes=2
).to(device)

x_test = torch.randint(
    0, 2, (4, 32),
    device=device
)

logits, states = model_v3(x_test)

print("Input :", x_test.shape)
print("Logits:", logits.shape)
print("States:", states.shape)
print("Finite :", torch.isfinite(states).all().item())

Input : torch.Size([4, 32])
Logits: torch.Size([4, 32, 2])
States: torch.Size([4, 32, 32])
Finite : True


In [78]:
# Cell: Running parity dataset

torch.manual_seed(42)

N = 8000
T = 16

X_run = torch.randint(
    0,
    2,
    (N, T),
    device=device
)

# Cumulative number of 1s
cumulative_ones = X_run.cumsum(dim=1)

# Running parity:
# 0 = even
# 1 = odd
y_run = cumulative_ones % 2

print("X:", X_run.shape)
print("Y:", y_run.shape)

print("\nExample:")
print("Input :", X_run[0].tolist())
print("Target:", y_run[0].tolist())

X: torch.Size([8000, 16])
Y: torch.Size([8000, 16])

Example:
Input : [1, 1, 0, 0, 1, 1, 0, 0, 0, 1, 1, 1, 1, 0, 1, 0]
Target: [1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 1]


In [79]:
# Cell: Train V3 on running parity

model_run = RLTModelV3(
    vocab_size=2,
    d_model=32,
    n_heads=4,
    ff_dim=128,
    window_size=8,
    max_seq_len=128,
    num_classes=2
).to(device)

optimizer_run = torch.optim.AdamW(
    model_run.parameters(),
    lr=3e-4,
    weight_decay=1e-2
)

batch_size = 64
steps = 1000

model_run.train()

for step in range(steps):

    idx = torch.randint(
        0,
        X_run.size(0),
        (batch_size,),
        device=device
    )

    x_batch = X_run[idx]
    y_batch = y_run[idx]

    # [B, T, 2]
    logits, states = model_run(x_batch)

    # Flatten token predictions
    loss = F.cross_entropy(
        logits.reshape(-1, 2),
        y_batch.reshape(-1)
    )

    optimizer_run.zero_grad(set_to_none=True)
    loss.backward()
    optimizer_run.step()

    if step % 100 == 0:

        with torch.no_grad():
            pred = logits.argmax(dim=-1)

            acc = (
                pred == y_batch
            ).float().mean().item() * 100

        print(
            f"Step {step:4d} | "
            f"Loss {loss.item():.4f} | "
            f"Accuracy {acc:.2f}%"
        )

Step    0 | Loss 0.7611 | Accuracy 51.46%
Step  100 | Loss 0.6221 | Accuracy 56.84%
Step  200 | Loss 0.5470 | Accuracy 63.96%
Step  300 | Loss 0.5306 | Accuracy 62.50%
Step  400 | Loss 0.4380 | Accuracy 71.09%
Step  500 | Loss 0.0062 | Accuracy 100.00%
Step  600 | Loss 0.0028 | Accuracy 100.00%
Step  700 | Loss 0.0018 | Accuracy 100.00%
Step  800 | Loss 0.0013 | Accuracy 100.00%
Step  900 | Loss 0.0010 | Accuracy 100.00%


**Does the learned recurrent computation continue working when we give it a sequence longer than anything it saw during training?**

In [80]:
# Cell: Evaluate running parity at training length

model_run.eval()

with torch.no_grad():

    logits_16, _ = model_run(X_run)

    pred_16 = logits_16.argmax(dim=-1)

    acc_16 = (
        pred_16 == y_run
    ).float().mean().item() * 100

print(f"Length 16 Accuracy: {acc_16:.2f}%")

Length 16 Accuracy: 100.00%


In [81]:
# Cell: Generate fresh longer sequences

def make_running_parity_data(N, T, device):
    X = torch.randint(
        0, 2,
        (N, T),
        device=device
    )

    Y = X.cumsum(dim=1) % 2

    return X, Y


X_32, y_32 = make_running_parity_data(
    N=2000,
    T=32,
    device=device
)

X_64, y_64 = make_running_parity_data(
    N=2000,
    T=64,
    device=device
)

print("Length 32:", X_32.shape, y_32.shape)
print("Length 64:", X_64.shape, y_64.shape)

Length 32: torch.Size([2000, 32]) torch.Size([2000, 32])
Length 64: torch.Size([2000, 64]) torch.Size([2000, 64])


In [82]:
# Cell: Evaluate length generalization

model_run.eval()

with torch.no_grad():

    # Length 16
    logits_16, _ = model_run(X_run)
    pred_16 = logits_16.argmax(dim=-1)
    acc_16 = (
        pred_16 == y_run
    ).float().mean().item() * 100

    # Length 32
    logits_32, _ = model_run(X_32)
    pred_32 = logits_32.argmax(dim=-1)
    acc_32 = (
        pred_32 == y_32
    ).float().mean().item() * 100

    # Length 64
    logits_64, _ = model_run(X_64)
    pred_64 = logits_64.argmax(dim=-1)
    acc_64 = (
        pred_64 == y_64
    ).float().mean().item() * 100

print(f"Length 16 → {acc_16:.2f}%")
print(f"Length 32 → {acc_32:.2f}%")
print(f"Length 64 → {acc_64:.2f}%")

Length 16 → 100.00%
Length 32 → 85.40%
Length 64 → 68.73%


In [83]:
# Cell: Accuracy by position

model_run.eval()

with torch.no_grad():
    logits_64, _ = model_run(X_64)
    pred_64 = logits_64.argmax(dim=-1)

position_acc = (pred_64 == y_64).float().mean(dim=0) * 100

for i, acc in enumerate(position_acc):
    print(f"Position {i+1:2d} → {acc.item():.2f}%")

Position  1 → 100.00%
Position  2 → 100.00%
Position  3 → 100.00%
Position  4 → 100.00%
Position  5 → 100.00%
Position  6 → 100.00%
Position  7 → 100.00%
Position  8 → 100.00%
Position  9 → 100.00%
Position 10 → 100.00%
Position 11 → 100.00%
Position 12 → 100.00%
Position 13 → 100.00%
Position 14 → 100.00%
Position 15 → 100.00%
Position 16 → 100.00%
Position 17 → 85.90%
Position 18 → 85.00%
Position 19 → 85.30%
Position 20 → 85.00%
Position 21 → 83.90%
Position 22 → 65.85%
Position 23 → 65.40%
Position 24 → 65.40%
Position 25 → 65.30%
Position 26 → 65.20%
Position 27 → 64.70%
Position 28 → 61.65%
Position 29 → 62.40%
Position 30 → 62.45%
Position 31 → 62.20%
Position 32 → 52.05%
Position 33 → 53.15%
Position 34 → 54.05%
Position 35 → 54.00%
Position 36 → 53.15%
Position 37 → 53.45%
Position 38 → 53.05%
Position 39 → 52.90%
Position 40 → 52.90%
Position 41 → 52.80%
Position 42 → 52.80%
Position 43 → 52.80%
Position 44 → 52.85%
Position 45 → 52.85%
Position 46 → 53.45%
Position 47 → 53.2

Your model learned:
“How to compute running parity for the first 16 steps”
but it has not learned a perfectly reusable state-transition rule that can continue indefinitely.
After position 16, the recurrent state begins drifting away from the representation needed for the parity calculation. Once that internal representation becomes unreliable enough, the predictions approach ~50%.
This is especially interesting because our RLT architecture does recurrently reuse the same transition block. The problem is therefore not that the model lacks recurrence—it has recurrence—but that training only supervised the final running-parity sequence at length 16.
That distinction matters. Research on looped Transformers has found that simply having a loop does not guarantee length generalization; training procedures that encourage reusable iterative computation can substantially improve it.

In [84]:
# Cell: RLTModelV4 — remove absolute positional embeddings

class RLTModelV4(nn.Module):
    def __init__(
        self,
        vocab_size,
        d_model=32,
        n_heads=4,
        ff_dim=128,
        window_size=8,
        max_seq_len=128,
        num_classes=None
    ):
        super().__init__()

        if num_classes is None:
            num_classes = vocab_size

        self.d_model = d_model
        self.max_seq_len = max_seq_len

        # No positional embedding
        self.token_embedding = nn.Embedding(vocab_size, d_model)

        self.encoder = RLTEncoder(
            d_model=d_model,
            n_heads=n_heads,
            ff_dim=ff_dim
        )

        self.transition = RLTDecoderTransitionV2(
            d_model=d_model,
            n_heads=n_heads,
            ff_dim=ff_dim,
            window_size=window_size
        )

        self.output_norm = RMSNorm(d_model)
        self.output_head = nn.Linear(d_model, num_classes)

    def forward(self, input_ids):

        B, T = input_ids.shape

        assert T <= self.max_seq_len

        # Token embedding only
        x = self.token_embedding(input_ids)

        memory = self.encoder(x)

        s_prev = torch.zeros(
            B,
            self.d_model,
            device=input_ids.device,
            dtype=x.dtype
        )

        cache = None
        states = []

        for t in range(T):

            e_t = memory[:, t, :]
            memory_prefix = memory[:, :t+1, :]

            s_t, cache = self.transition(
                e_t,
                s_prev,
                memory_prefix,
                cache
            )

            states.append(s_t)
            s_prev = s_t

        states = torch.stack(states, dim=1)

        normalized_states = self.output_norm(states)

        logits = self.output_head(normalized_states)

        return logits, states


model_run_v4 = RLTModelV4(
    vocab_size=2,
    d_model=32,
    n_heads=4,
    ff_dim=128,
    window_size=8,
    max_seq_len=128,
    num_classes=2
).to(device)

print("Parameters:", sum(p.numel() for p in model_run_v4.parameters()))

Parameters: 32802


In [85]:
# Cell: Train RLTModelV4
model_run_v4.train()

optimizer_v4 = torch.optim.AdamW(
    model_run_v4.parameters(),
    lr=3e-4,
    weight_decay=1e-2
)

batch_size = 64
steps = 1000

for step in range(steps):

    idx = torch.randint(
        0, X_run.shape[0],
        (batch_size,),
        device=device
    )

    X_batch = X_run[idx]
    y_batch = y_run[idx]

    logits, _ = model_run_v4(X_batch)

    loss = F.cross_entropy(
        logits.reshape(-1, 2),
        y_batch.reshape(-1)
    )

    optimizer_v4.zero_grad()
    loss.backward()
    optimizer_v4.step()

    if step % 100 == 0:
        with torch.no_grad():
            acc = (
                logits.argmax(dim=-1) == y_batch
            ).float().mean().item() * 100

        print(
            f"Step {step:4d} | "
            f"Loss {loss.item():.4f} | "
            f"Accuracy {acc:.2f}%"
        )

Step    0 | Loss 0.8159 | Accuracy 48.44%
Step  100 | Loss 0.6492 | Accuracy 56.35%
Step  200 | Loss 0.6278 | Accuracy 58.50%
Step  300 | Loss 0.6002 | Accuracy 56.64%
Step  400 | Loss 0.4802 | Accuracy 71.97%
Step  500 | Loss 0.0223 | Accuracy 100.00%
Step  600 | Loss 0.0075 | Accuracy 100.00%
Step  700 | Loss 0.0045 | Accuracy 100.00%
Step  800 | Loss 0.0031 | Accuracy 100.00%
Step  900 | Loss 0.0023 | Accuracy 100.00%


In [86]:
# Cell: Evaluate V4 at lengths 16, 32, 64

model_run_v4.eval()

with torch.no_grad():

    logits_16_v4, _ = model_run_v4(X_run)
    pred_16_v4 = logits_16_v4.argmax(dim=-1)
    acc_16_v4 = (
        (pred_16_v4 == y_run)
        .float()
        .mean()
        .item() * 100
    )

    logits_32_v4, _ = model_run_v4(X_32)
    pred_32_v4 = logits_32_v4.argmax(dim=-1)
    acc_32_v4 = (
        (pred_32_v4 == y_32)
        .float()
        .mean()
        .item() * 100
    )

    logits_64_v4, _ = model_run_v4(X_64)
    pred_64_v4 = logits_64_v4.argmax(dim=-1)
    acc_64_v4 = (
        (pred_64_v4 == y_64)
        .float()
        .mean()
        .item() * 100
    )

print(f"V4 Length 16 → {acc_16_v4:.2f}%")
print(f"V4 Length 32 → {acc_32_v4:.2f}%")
print(f"V4 Length 64 → {acc_64_v4:.2f}%")

V4 Length 16 → 100.00%
V4 Length 32 → 99.99%
V4 Length 64 → 99.57%


In [87]:
# Cell: Create long-context V4

model_run_v4_long = RLTModelV4(
    vocab_size=2,
    d_model=32,
    n_heads=4,
    ff_dim=128,
    window_size=8,
    max_seq_len=512,
    num_classes=2
).to(device)

print(
    "Parameters:",
    sum(p.numel() for p in model_run_v4_long.parameters())
)

Parameters: 32802


In [88]:
# Cell: Transfer trained V4 weights to long-context V4

model_run_v4_long.load_state_dict(model_run_v4.state_dict())

print("Weights transferred successfully.")
print(
    "Parameters:",
    sum(p.numel() for p in model_run_v4_long.parameters())
)

Weights transferred successfully.
Parameters: 32802


In [89]:
# Cell: Generate fresh long-context running-parity data

def make_running_parity_data(N, T, device):
    X = torch.randint(0, 2, (N, T), device=device)
    Y = X.cumsum(dim=1) % 2
    return X, Y


X_128, y_128 = make_running_parity_data(1000, 128, device)
X_256, y_256 = make_running_parity_data(1000, 256, device)
X_512, y_512 = make_running_parity_data(500, 512, device)

print("128:", X_128.shape)
print("256:", X_256.shape)
print("512:", X_512.shape)

128: torch.Size([1000, 128])
256: torch.Size([1000, 256])
512: torch.Size([500, 512])


In [90]:
# Cell: Long-context evaluation

model_run_v4_long.eval()

with torch.no_grad():

    logits_128, _ = model_run_v4_long(X_128)
    acc_128 = (
        (logits_128.argmax(dim=-1) == y_128)
        .float()
        .mean()
        .item() * 100
    )

    logits_256, _ = model_run_v4_long(X_256)
    acc_256 = (
        (logits_256.argmax(dim=-1) == y_256)
        .float()
        .mean()
        .item() * 100
    )

    logits_512, _ = model_run_v4_long(X_512)
    acc_512 = (
        (logits_512.argmax(dim=-1) == y_512)
        .float()
        .mean()
        .item() * 100
    )

print(f"Length 128 → {acc_128:.2f}%")
print(f"Length 256 → {acc_256:.2f}%")
print(f"Length 512 → {acc_512:.2f}%")

Length 128 → 98.67%
Length 256 → 95.67%
Length 512 → 91.48%


In [91]:
# Cell: Position-wise accuracy at length 512

model_run_v4_long.eval()

with torch.no_grad():
    logits_512, _ = model_run_v4_long(X_512)
    pred_512 = logits_512.argmax(dim=-1)

position_acc_512 = (
    (pred_512 == y_512)
    .float()
    .mean(dim=0) * 100
)

# Print every 16th position
for i in range(0, 512, 16):
    end = min(i + 16, 512)
    avg = position_acc_512[i:end].mean().item()
    print(f"Positions {i+1:3d}-{end:3d} → {avg:.2f}%")

Positions   1- 16 → 100.00%
Positions  17- 32 → 99.84%
Positions  33- 48 → 99.20%
Positions  49- 64 → 98.65%
Positions  65- 80 → 97.80%
Positions  81- 96 → 96.85%
Positions  97-112 → 96.01%
Positions 113-128 → 95.00%
Positions 129-144 → 94.19%
Positions 145-160 → 93.90%
Positions 161-176 → 93.00%
Positions 177-192 → 92.31%
Positions 193-208 → 91.53%
Positions 209-224 → 91.33%
Positions 225-240 → 91.24%
Positions 241-256 → 91.40%
Positions 257-272 → 91.59%
Positions 273-288 → 90.70%
Positions 289-304 → 89.97%
Positions 305-320 → 89.64%
Positions 321-336 → 89.46%
Positions 337-352 → 89.13%
Positions 353-368 → 88.19%
Positions 369-384 → 88.03%
Positions 385-400 → 87.75%
Positions 401-416 → 87.25%
Positions 417-432 → 86.41%
Positions 433-448 → 86.04%
Positions 449-464 → 85.97%
Positions 465-480 → 85.53%
Positions 481-496 → 85.07%
Positions 497-512 → 84.44%
